# Dashboard Demo

Generate the dataset, create interactive charts with filters, and build the final dashboard in `dashboard.html`.

In [100]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np
import json
from collections import Counter

## 1. Load the Dataset

In [101]:
# ============================================
# CONFIGURE HERE: GitHub repo info
# ============================================
github_user = "elvaramosglz"
github_repo = "dashboard-demo"
github_branch = "main"
csv_filename = "dataset.csv"
dataset_encoding = "mac_roman"  # this file is saved as mac_roman, not UTF-8 (e.g. "Maracan\u00e3" breaks under utf-8)

# Built automatically \u2014 don't need to edit this line
dataset_url = f"https://raw.githubusercontent.com/{github_user}/{github_repo}/{github_branch}/{csv_filename}"

# Reads straight from GitHub, so anyone who opens this notebook gets the same
# CSV you have in the repo \u2014 no need to have the file saved locally
df_quality = pd.read_csv(dataset_url, encoding=dataset_encoding)
df_quality.head()

,MONTH,CITY,TRIPS,GROSS BOOKINGS (GB),SHOPPING SESSIONS,RIDERS WITH SESSIONS,REQUESTING SESSIONS,REQUESTS,ACTIVE DRIVERS,SUPPLY HOURS,...,RS/SS,C/SS,ACCEPTANCE RATE,DRIVER CANCELATION,SUPPLY UTILIZATION,NPI,DRIVER PAYMENTS (AS % OF GB),RIDER INCENTIVES AND PROMOTIONS (AS % OF GB),DRIVER INCENTIVES AND PROMOTIONS (AS % OF GB),TAKE RATE (AS % OF GB)
0,2022-01,Vila Belmiro,"3,005,706","56,529,569","5,283,006","711,432","3,257,826","3,844,423","18,795","1,564,816",...,61.7%,56.9%,23.0%,20.0%,63.0%,1.03,73.5%,5.7%,5.1%,14.2%
1,2022-01,Rose Bowl,"1,533,078","25,315,829","2,692,591","406,844","1,638,051","1,859,779","8,742","824,750",...,60.8%,56.9%,34.0%,15.0%,59.0%,0.98,76.3%,-3.3%,4.3%,20.5%
2,2022-01,San Siro,"8,108,570","155,667,588","14,385,079","1,609,850","8,570,326","9,515,128","50,056","4,187,961",...,59.6%,56.4%,44.0%,22.0%,73.0%,1.02,78.8%,4.9%,3.6%,10.6%
3,2022-01,Lusail,"2,163,550","38,225,180","3,939,724","493,960","2,348,175","2,859,544","10,845","1,074,968",...,59.6%,54.9%,25.0%,20.0%,70.0%,1.05,76.8%,6.2%,3.6%,11.8%
4,2022-01,Old Trafford,"11,042,462","206,436,144","19,113,887","2,553,885","11,801,188","13,152,423","63,551","5,202,255",...,61.7%,57.8%,46.0%,27.0%,77.0%,1.05,75.6%,-1.4%,3.4%,17.0%


## 2. Dashboard \u2014 Metric Over Time base table\n\nBase table (`CITY | MONTH | METRIC`) that will feed the Metric Over Time chart on `dashboard.html`. Built directly from the dataset loaded above.

In [102]:
# ============================================
# Long-format base table for the "Metric Over Time" chart:
# one row per (CITY, MONTH, METRIC) combination, straight from
# the real CSV loaded above \u2014 nothing invented here.
# ============================================
id_columns = ["MONTH", "CITY"]
metric_columns = [c for c in df_quality.columns if c not in id_columns]

metric_over_time = df_quality.melt(
    id_vars=id_columns,
    value_vars=metric_columns,
    var_name="METRIC",
    value_name="VALUE",
)

n_cities = df_quality["CITY"].nunique()
n_months = df_quality["MONTH"].nunique()
n_metrics = len(metric_columns)

print(f"{n_cities} cities \u00d7 {n_months} months \u00d7 {n_metrics} metrics = {len(metric_over_time)} rows")
print("Cities:", sorted(df_quality["CITY"].unique().tolist()))
print("Months:", sorted(df_quality["MONTH"].unique().tolist()))
print("Metrics:", metric_columns)

# the full base table (all cities, all months, all metrics) \u2014 this is
# what actually feeds the chart; pandas only truncates it for display
metric_over_time


8 cities × 21 months × 20 metrics = 3360 rows
Cities: ['Azteca', 'Lusail', 'Maracanã', 'Old Trafford', 'Rose Bowl', 'San Siro', 'Santiago Bernabeu', 'Vila Belmiro']
Months: ['2022-01', '2022-02', '2022-03', '2022-04', '2022-05', '2022-06', '2022-07', '2022-08', '2022-09', '2022-10', '2022-11', '2022-12', '2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06', '2023-07', '2023-08', '2023-09']
Metrics: ['TRIPS', 'GROSS BOOKINGS (GB)', 'SHOPPING SESSIONS', 'RIDERS WITH SESSIONS', 'REQUESTING SESSIONS', 'REQUESTS', 'ACTIVE DRIVERS', 'SUPPLY HOURS', 'C/R', 'C/RS', 'RS/SS', 'C/SS', 'ACCEPTANCE RATE', 'DRIVER CANCELATION', 'SUPPLY UTILIZATION', 'NPI', 'DRIVER PAYMENTS (AS % OF GB)', 'RIDER INCENTIVES AND PROMOTIONS (AS % OF GB)', 'DRIVER INCENTIVES AND PROMOTIONS (AS % OF GB)', 'TAKE RATE (AS % OF GB)']


,MONTH,CITY,METRIC,VALUE
0,2022-01,Vila Belmiro,TRIPS,"3,005,706"
1,2022-01,Rose Bowl,TRIPS,"1,533,078"
2,2022-01,San Siro,TRIPS,"8,108,570"
3,2022-01,Lusail,TRIPS,"2,163,550"
4,2022-01,Old Trafford,TRIPS,"11,042,462"
...,...,...,...,...
3355,2023-09,Lusail,TAKE RATE (AS % OF GB),14.7%
3356,2023-09,Old Trafford,TAKE RATE (AS % OF GB),11.9%
3357,2023-09,Maracanã,TAKE RATE (AS % OF GB),12.7%
3358,2023-09,Azteca,TAKE RATE (AS % OF GB),12.3%


## 3. Dashboard \u2014 Clean metric values for charting

In [103]:
# ============================================
# Clean VALUE into real numbers for charting.
# Most metric columns are stored as strings in the CSV (thousands
# separators like "3,005,706", or percentages like "78.2%"), so this
# just parses what's actually there \u2014 no values are invented.
# ============================================
def parse_metric_value(raw):
    if pd.isna(raw):
        return None
    text = str(raw).strip().replace(",", "").replace("%", "")
    try:
        return float(text)
    except ValueError:
        return None

metric_over_time["VALUE_NUM"] = metric_over_time["VALUE"].apply(parse_metric_value)
metric_over_time[["CITY", "MONTH", "METRIC", "VALUE", "VALUE_NUM"]].head()


,CITY,MONTH,METRIC,VALUE,VALUE_NUM
0,Vila Belmiro,2022-01,TRIPS,"3,005,706",3005706.0
1,Rose Bowl,2022-01,TRIPS,"1,533,078",1533078.0
2,San Siro,2022-01,TRIPS,"8,108,570",8108570.0
3,Lusail,2022-01,TRIPS,"2,163,550",2163550.0
4,Old Trafford,2022-01,TRIPS,"11,042,462",11042462.0


## 4. Dashboard \u2014 Build the Metric Over Time chart data

In [104]:
# ============================================
# CONFIGURE HERE: default metric shown when the page first loads
# ============================================
default_metric = "TRIPS"

months_sorted = sorted(df_quality["MONTH"].unique().tolist())
cities_sorted = sorted(df_quality["CITY"].unique().tolist())

chart_data = {"months": months_sorted, "cities": cities_sorted, "metrics": {}}

# A metric counts as a percentage if its raw CSV values actually contain a
# "%" sign \u2014 checked against the real data, not assumed by name.
percent_metrics = [
    metric
    for metric in metric_columns
    if metric_over_time.loc[metric_over_time["METRIC"] == metric, "VALUE"]
    .astype(str)
    .str.contains("%")
    .any()
]
chart_data["percent_metrics"] = percent_metrics

# Real min/max per percent metric, straight from the data \u2014 used to
# scale the Y axis to what actually happens instead of a flat 0-100.
chart_data["metric_ranges"] = {
    metric: {
        "min": metric_over_time.loc[metric_over_time["METRIC"] == metric, "VALUE_NUM"].min(),
        "max": metric_over_time.loc[metric_over_time["METRIC"] == metric, "VALUE_NUM"].max(),
    }
    for metric in percent_metrics
}

for metric in metric_columns:
    metric_slice = metric_over_time[metric_over_time["METRIC"] == metric]
    per_city = {}
    for city in cities_sorted:
        series = metric_slice[metric_slice["CITY"] == city].set_index("MONTH")["VALUE_NUM"]
        per_city[city] = [series.get(m) for m in months_sorted]
    chart_data["metrics"][metric] = per_city

chart_data_json = json.dumps(chart_data)
print(f"Chart data ready: {len(months_sorted)} months \u00d7 {len(cities_sorted)} cities \u00d7 {len(metric_columns)} metrics")
print("Percent metrics:", percent_metrics)
for m in percent_metrics:
    r = chart_data["metric_ranges"][m]
    print(f"  {m}: min={r['min']}, max={r['max']}")


Chart data ready: 21 months × 8 cities × 20 metrics
Percent metrics: ['C/R', 'C/RS', 'RS/SS', 'C/SS', 'ACCEPTANCE RATE', 'DRIVER CANCELATION', 'SUPPLY UTILIZATION', 'DRIVER PAYMENTS (AS % OF GB)', 'RIDER INCENTIVES AND PROMOTIONS (AS % OF GB)', 'DRIVER INCENTIVES AND PROMOTIONS (AS % OF GB)', 'TAKE RATE (AS % OF GB)']
  C/R: min=59.7, max=90.7
  C/RS: min=81.0, max=96.8
  RS/SS: min=52.4, max=67.8
  C/SS: min=46.5, max=65.3
  ACCEPTANCE RATE: min=17.0, max=46.0
  DRIVER CANCELATION: min=13.0, max=32.0
  SUPPLY UTILIZATION: min=55.0, max=81.0
  DRIVER PAYMENTS (AS % OF GB): min=69.0, max=79.9
  RIDER INCENTIVES AND PROMOTIONS (AS % OF GB): min=-8.5, max=17.4
  DRIVER INCENTIVES AND PROMOTIONS (AS % OF GB): min=0.0, max=9.1
  TAKE RATE (AS % OF GB): min=4.2, max=24.3


## 5. Build the Dashboard (`dashboard.html`)

In [105]:
# ============================================
# CONFIGURE HERE: colors (same palette as data_quality.html) and the
# per-city line colors for the chart
# ============================================
uber_blue = "#2171EC"
uber_blue_dark = "#1257B8"
uber_ink = "#121212"
uber_muted = "#6B6F76"
uber_faint = "#ADADAD"
uber_canvas = "#F1F2F4"
uber_surface = "#FFFFFF"
uber_border = "#DEE1E6"

city_palette = [
    "#2171EC",  # blue (brand)
    "#0EA5C4",  # teal / cyan
    "#7C5CFC",  # purple / indigo
    "#E0900B",  # amber
    "#E14F45",  # coral / red
    "#2E9E4F",  # green
    "#D6478B",  # pink / magenta
    "#6B7280",  # slate gray
]

metric_options_html = "".join(
    f'<option value="{m}"{" selected" if m == default_metric else ""}>{m}</option>'
    for m in metric_columns
)
city_checkbox_items_html = "".join(
    f'<label><input type="checkbox" class="city-checkbox" value="{c}" checked> {c}</label>'
    for c in cities_sorted
)

dashboard_html = f'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Dashboard</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<style>
  :root {{
    --canvas: {uber_canvas};
    --surface: {uber_surface};
    --border: {uber_border};
    --ink: {uber_ink};
    --muted: {uber_muted};
    --faint: {uber_faint};
    --blue: {uber_blue};
    --blue-dark: {uber_blue_dark};
    --ok: #1F8B4C;
    --ok-bg: #E6F4EC;
    --bad: #C4291C;
    --bad-bg: #FBEAE8;
    --warn: #B7791F;
    --warn-bg: #FCF6EA;
    --radius: 10px;
    --radius-sm: 6px;
    --sans: -apple-system, BlinkMacSystemFont, \'Segoe UI\', Roboto, Helvetica, Arial, sans-serif;
    --mono: ui-monospace, SFMono-Regular, \'Roboto Mono\', Menlo, Consolas, monospace;
  }}

  * {{ box-sizing: border-box; margin: 0; padding: 0; font-family: var(--sans); }}
  body {{ background: var(--canvas); color: var(--ink); padding: 40px 32px; }}
  .wrap {{ max-width: 1400px; margin: 0 auto; }}
  #metric-over-time, #revenue-by-product, #units-distribution {{ scroll-margin-top: 24px; }}

  h2 {{
    position: relative;
    color: #FFFFFF;
    background: var(--ink);
    margin: -28px -32px 22px;
    padding: 22px 32px 24px;
    text-align: center;
    text-transform: uppercase;
    font-size: 21px;
    font-weight: 800;
    letter-spacing: 0.08em;
    border-radius: var(--radius) var(--radius) 0 0;
  }}
  h2::before {{
    content: "";
    position: absolute;
    left: 50%;
    bottom: 12px;
    transform: translateX(-50%);
    width: 40px;
    height: 3px;
    background: var(--blue);
    border-radius: 2px;
  }}
  .description {{ color: var(--muted); font-size: 13px; margin: 0 0 20px; line-height: 1.5; text-align: left; }}

  .dashboard-grid {{
    display: grid;
    grid-template-columns: 1fr 1fr;
    grid-template-rows: auto 1fr auto;
    gap: 24px;
    min-height: calc(100vh - 80px);
  }}

  .panel {{
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: var(--radius);
    box-shadow: 0 1px 2px rgba(18, 18, 18, 0.04);
    padding: 28px 32px;
    display: flex;
    flex-direction: column;
    overflow: hidden;
  }}
  .panel-left {{ grid-column: 1 / 3; grid-row: 1; min-height: 560px; }}
  .panel-top-right {{ grid-column: 1; grid-row: 2; }}
  .panel-bottom-right {{ grid-column: 2; grid-row: 2; }}
  .panel-yoy {{ grid-column: 1 / 3; grid-row: 3; }}

  .chart-placeholder {{
    flex: 1;
    display: flex;
    align-items: center;
    justify-content: center;
    text-align: center;
    border: 1px dashed var(--border);
    border-radius: var(--radius-sm);
    color: var(--muted);
    font-size: 13px;
    line-height: 1.5;
    padding: 24px;
  }}

  .chart-controls {{ display: flex; gap: 28px; flex-wrap: wrap; margin-bottom: 16px; }}
  .control-group label {{ font-size: 11px; text-transform: uppercase; letter-spacing: 0.04em; font-weight: 700; color: var(--muted); display: block; margin-bottom: 8px; }}
  select {{
    padding: 9px 34px 9px 12px;
    border-radius: var(--radius-sm);
    border: 1px solid var(--border);
    font-size: 13px;
    font-family: var(--sans);
    background: var(--surface);
    color: var(--ink);
  }}
  .multiselect {{ position: relative; }}
  .multiselect-toggle {{
    min-width: 200px;
    text-align: left;
    padding: 9px 34px 9px 12px;
    border-radius: var(--radius-sm);
    border: 1px solid var(--border);
    background: var(--surface);
    font-size: 13px;
    font-family: var(--sans);
    color: var(--ink);
    cursor: pointer;
    position: relative;
  }}
  .multiselect-toggle::after {{
    content: "\\25BE";
    position: absolute;
    right: 12px;
    top: 50%;
    transform: translateY(-50%);
    color: var(--muted);
    font-size: 11px;
  }}
  .multiselect-panel {{
    position: absolute;
    top: calc(100% + 6px);
    left: 0;
    z-index: 10;
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: var(--radius-sm);
    box-shadow: 0 4px 14px rgba(18, 18, 18, 0.12);
    padding: 8px;
    display: flex;
    flex-direction: column;
    gap: 2px;
    min-width: 220px;
  }}
  .multiselect-panel[hidden] {{ display: none; }}
  .multiselect-panel label {{
    display: flex;
    align-items: center;
    gap: 8px;
    padding: 6px 8px;
    border-radius: 4px;
    font-size: 13px;
    font-weight: 400;
    text-transform: none;
    letter-spacing: normal;
    color: var(--ink);
    cursor: pointer;
  }}
  .multiselect-panel label:hover {{ background: var(--canvas); }}
  .chart-canvas-wrap {{ flex: 1; position: relative; min-height: 260px; }}
</style>
</head>
<body>
  <div class="wrap">
    <div class="dashboard-grid">
      <div class="panel panel-left" id="metric-over-time">
        <h2>Metric Over Time</h2>
        <p class="description">This chart tracks a single metric across every month in the dataset, with one line per city. Use it to compare how a metric evolved over time between markets, spot trends, seasonal patterns, or outliers in specific cities. Choose the metric and the cities to compare below.</p>
        <div class="chart-controls">
          <div class="control-group">
            <label for="metric-select">Metric</label>
            <select id="metric-select">{metric_options_html}</select>
          </div>
          <div class="control-group">
            <label>Cities</label>
            <div class="multiselect" id="city-multiselect">
              <button type="button" class="multiselect-toggle" id="city-toggle">All cities</button>
              <div class="multiselect-panel" id="city-panel" hidden>{city_checkbox_items_html}</div>
            </div>
          </div>
        </div>
        <div class="chart-canvas-wrap">
          <canvas id="metric-chart"></canvas>
        </div>
      </div>
      <div class="panel panel-top-right" id="revenue-by-product">
        <h2>Revenue By Product</h2>
        <div class="chart-placeholder">Chart pending connection to real data.</div>
      </div>
      <div class="panel panel-bottom-right" id="units-distribution">
        <h2>Units Distribution</h2>
        <div class="chart-placeholder">Chart pending connection to real data.</div>
      </div>
      <div class="panel panel-yoy" id="year-over-year">
        <!-- YOY_CONTENT_START -->
        <div class="chart-placeholder">Year-over-Year Analysis pending.</div>
        <!-- YOY_CONTENT_END -->
      </div>
    </div>
  </div>

<script>
  const chartData = {chart_data_json};
  const palette = {json.dumps(city_palette)};
  const metricSelect = document.getElementById(\'metric-select\');
  const cityCheckboxes = Array.from(document.querySelectorAll(\'.city-checkbox\'));
  const cityToggle = document.getElementById(\'city-toggle\');
  const cityPanel = document.getElementById(\'city-panel\');
  const ctx = document.getElementById(\'metric-chart\').getContext(\'2d\');
  let chart;

  function selectedCities() {{
    return cityCheckboxes.filter(function(cb) {{ return cb.checked; }}).map(function(cb) {{ return cb.value; }});
  }}

  function updateCityToggleLabel() {{
    const selected = selectedCities();
    if (selected.length === 0) {{
      cityToggle.textContent = \'No cities selected\';
    }} else if (selected.length === chartData.cities.length) {{
      cityToggle.textContent = \'All cities (\' + selected.length + \')\';
    }} else if (selected.length <= 2) {{
      cityToggle.textContent = selected.join(\', \');
    }} else {{
      cityToggle.textContent = selected.length + \' cities selected\';
    }}
  }}

  cityToggle.addEventListener(\'click\', function(event) {{
    event.stopPropagation();
    cityPanel.hidden = !cityPanel.hidden;
  }});

  document.addEventListener(\'click\', function(event) {{
    if (!cityPanel.hidden && !cityPanel.contains(event.target) && event.target !== cityToggle) {{
      cityPanel.hidden = true;
    }}
  }});

  function yAxisOptions(metric) {{
    if (chartData.percent_metrics.includes(metric)) {{
      const range = chartData.metric_ranges[metric];
      return {{
        min: range.min,
        max: range.max,
        ticks: {{ callback: function(value) {{ return value.toFixed(1) + \'%\'; }} }},
      }};
    }}
    return {{ beginAtZero: false }};
  }}

  function buildDatasets(metric, cities) {{
    return cities.map(function(city) {{
      const idx = chartData.cities.indexOf(city);
      const color = palette[idx % palette.length];
      return {{
        label: city,
        data: chartData.metrics[metric][city],
        borderColor: color,
        backgroundColor: color,
        borderWidth: 2,
        pointRadius: 2,
        tension: 0.25,
        spanGaps: true,
      }};
    }});
  }}

  function renderChart() {{
    const metric = metricSelect.value;
    const cities = selectedCities();
    const datasets = buildDatasets(metric, cities);
    const isPercent = chartData.percent_metrics.includes(metric);
    const yOptions = yAxisOptions(metric);
    yOptions.title = {{
      display: true,
      text: metric + (isPercent ? \' (%)\' : \'\'),
      color: \'#6B6F76\',
      font: {{ size: 12, weight: \'600\' }},
    }};
    yOptions.grid = {{ color: \'#DEE1E6\' }};
    yOptions.ticks = Object.assign({{ color: \'#6B6F76\' }}, yOptions.ticks || {{}});

    const chartTitle = (metric + \' OVER TIME\').toUpperCase();

    const tooltipLabel = function(item) {{
      return item.dataset.label + \': \' + item.formattedValue + (isPercent ? \'%\' : \'\');
    }};

    if (chart) {{
      chart.data.labels = chartData.months;
      chart.data.datasets = datasets;
      chart.options.scales.y = yOptions;
      chart.options.plugins.title.text = chartTitle;
      chart.options.plugins.tooltip.callbacks.label = tooltipLabel;
      chart.update();
    }} else {{
      chart = new Chart(ctx, {{
        type: \'line\',
        data: {{ labels: chartData.months, datasets: datasets }},
        options: {{
          responsive: true,
          maintainAspectRatio: false,
          interaction: {{ mode: \'index\', intersect: false }},
          layout: {{ padding: {{ top: 8, bottom: 8 }} }},
          scales: {{
            x: {{
              title: {{ display: true, text: \'Month\', color: \'#6B6F76\', font: {{ size: 12, weight: \'600\' }} }},
              grid: {{ color: \'#DEE1E6\' }},
              ticks: {{ color: \'#6B6F76\' }},
            }},
            y: yOptions,
          }},
          plugins: {{
            title: {{
              display: true,
              text: chartTitle,
              color: \'#121212\',
              font: {{ size: 16, weight: \'700\' }},
              padding: {{ bottom: 12 }},
            }},
            legend: {{
              labels: {{ color: \'#121212\', font: {{ size: 12 }}, usePointStyle: true, pointStyle: \'circle\', padding: 20, boxHeight: 8 }},
              padding: {{ bottom: 40 }},
            }},
            tooltip: {{
              backgroundColor: \'#121212\',
              titleFont: {{ size: 12, weight: \'600\' }},
              bodyFont: {{ size: 12 }},
              padding: 10,
              cornerRadius: 6,
              callbacks: {{ label: tooltipLabel }},
            }},
          }},
        }},
      }});
    }}
  }}

  metricSelect.addEventListener(\'change\', renderChart);
  cityCheckboxes.forEach(function(cb) {{
    cb.addEventListener(\'change\', function() {{
      updateCityToggleLabel();
      renderChart();
    }});
  }});
  updateCityToggleLabel();
  renderChart();

  // YOY_JS_START
  // YOY_JS_END
</script>
</body>
</html>
'''

with open("dashboard.html", "w", encoding="utf-8") as f:
    f.write(dashboard_html)

print("dashboard.html generated \u2014 Metric Over Time chart wired to real CSV data")


dashboard.html generated — Metric Over Time chart wired to real CSV data


## 6. Year-over-Year Analysis \u2014 calculation engine\n\nCompares a period against the same calendar period one year earlier. Volume metrics are reported as % growth; rate/ratio metrics are reported as percentage-point (pp) change. `All Cities` aggregates numerators/denominators first, then computes the ratio \u2014 it never averages each city's percentage.

In [106]:
# ============================================
# CONFIGURE HERE: funnel definition and ratio components
# ============================================

# Fixed order shown when the user selects "Full Funnel"
#funnel_metrics = ["SHOPPING SESSIONS", "RS/SS", "REQUESTING SESSIONS", "C/RS", "TRIPS", "C/SS"]

# Numerator / denominator for every ratio metric we can recompute from its
# real components (needed to aggregate "All Cities" correctly instead of
# averaging percentages).
ratio_components = {
    "RS/SS": ("REQUESTING SESSIONS", "SHOPPING SESSIONS"),
    "C/RS": ("TRIPS", "REQUESTING SESSIONS"),
    "C/R": ("TRIPS", "REQUESTS"),
    "C/SS": ("TRIPS", "SHOPPING SESSIONS"),
}

# `percent_metrics` was already detected earlier from the real CSV values
# (any column whose raw text contains "%"). Reused here as the official
# list of rate/ratio metrics \u2014 reported in percentage points (pp), not %.
ratio_metrics = set(percent_metrics)
volume_metrics = [m for m in metric_columns if m not in ratio_metrics]

# A clean, fully numeric copy of the dataset (reuses the same parser
# already used for the Metric Over Time chart \u2014 no separate logic).
df_numeric = df_quality.copy()
for col in metric_columns:
    df_numeric[col] = df_numeric[col].apply(parse_metric_value)

all_months_sorted = sorted(df_quality["MONTH"].unique().tolist())
all_cities_sorted = sorted(df_quality["CITY"].unique().tolist())


def prior_year_month(month):
    year, mo = month.split("-")
    return f"{int(year) - 1}-{mo}"


def filter_slice(months, cities=None):
    mask = df_numeric["MONTH"].isin(months)
    if cities:
        mask = mask & df_numeric["CITY"].isin(cities)
    return df_numeric[mask]


def aggregate_ratio(df_slice, metric):
    """Recomputes a ratio from its aggregated numerator/denominator
    (weighted), instead of averaging each city's percentage."""
    if metric in ratio_components:
        num_col, den_col = ratio_components[metric]
        denominator = df_slice[den_col].sum()
        if denominator == 0:
            return None
        return df_slice[num_col].sum() / denominator * 100
    # No decomposable components exist in the CSV for this rate (e.g.
    # ACCEPTANCE RATE, DRIVER CANCELATION). Falls back to a simple mean
    # across the selected rows \u2014 flagged explicitly in the UI as a
    # methodology limitation rather than silently presented as weighted.
    values = df_slice[metric].dropna()
    return values.mean() if not values.empty else None


def format_volume(value):
    if value is None:
        return "\u2013"
    abs_value = abs(value)
    if abs_value >= 1_000_000:
        return f"{value / 1_000_000:.1f}M"
    if abs_value >= 1_000:
        return f"{value / 1_000:.0f}K"
    return f"{value:.0f}"


def format_rate(value):
    return "\u2013" if value is None else f"{value:.1f}%"


def compute_yoy_row(metric, current_months, prior_months, cities=None):
    is_ratio = metric in ratio_metrics
    current_slice = filter_slice(current_months, cities)
    prior_slice = filter_slice(prior_months, cities)

    missing_prior_months = [m for m in prior_months if m not in all_months_sorted]
    if current_slice.empty or prior_slice.empty or missing_prior_months:
        return {
            "metric": metric, "is_ratio": is_ratio,
            "prior_display": "\u2013", "current_display": "\u2013",
            "change_display": "Not comparable \u2014 matching prior-year month is unavailable.",
        }

    if is_ratio:
        prior_value = aggregate_ratio(prior_slice, metric)
        current_value = aggregate_ratio(current_slice, metric)
        if prior_value is None or current_value is None:
            return {
                "metric": metric, "is_ratio": is_ratio,
                "prior_display": format_rate(prior_value), "current_display": format_rate(current_value),
                "change_display": "Not available \u2014 prior-year value is zero.",
            }
        change = current_value - prior_value
        sign = "+" if change >= 0 else "\u2212"
        return {
            "metric": metric, "is_ratio": is_ratio,
            "prior_value": prior_value, "current_value": current_value, "change": change,
            "prior_display": format_rate(prior_value), "current_display": format_rate(current_value),
            "change_display": f"{sign}{abs(change):.1f} pp",
        }
    else:
        prior_value = current_slice[metric].sum() * 0 + prior_slice[metric].sum()
        current_value = current_slice[metric].sum()
        if prior_value == 0:
            return {
                "metric": metric, "is_ratio": is_ratio,
                "prior_display": format_volume(prior_value), "current_display": format_volume(current_value),
                "change_display": "Not available \u2014 prior-year value is zero.",
            }
        change_pct = (current_value / prior_value - 1) * 100
        sign = "+" if change_pct >= 0 else "\u2212"
        return {
            "metric": metric, "is_ratio": is_ratio,
            "prior_value": prior_value, "current_value": current_value, "change": change_pct,
            "prior_display": format_volume(prior_value), "current_display": format_volume(current_value),
            "change_display": f"{sign}{abs(change_pct):.1f}%",
        }


def build_funnel_table(target_month, cities=None, metrics=None):
    """target_month: e.g. "2023-02". Compares it against the same month
    one year earlier, for the given metrics (defaults to the full funnel)."""
    metrics = metrics
    prior_month = prior_year_month(target_month)
    rows = [
        compute_yoy_row(metric, [target_month], [prior_month], cities)
        for metric in metrics
    ]
    return prior_month, rows




## 6b. Build the Year-Over-Year Analysis panel (`dashboard.html`)

Renders the funnel table, filters (Month / Cities / View / Metric Scope), narrative, and methodology note into `dashboard.html`, reusing the same real numeric data already embedded for the Metric Over Time chart.

In [107]:
from pathlib import Path

# ============================================
# CONFIGURE HERE: which metrics get a business-logic color (spec section 8)
# ============================================
yoy_polarity = {
    "TRIPS": "good_up",
    "C/SS": "good_up",
    "SHOPPING SESSIONS": "good_up",
    "REQUESTING SESSIONS": "good_up",
    "REQUESTS": "good_up",
    "RS/SS": "good_up",
    "C/RS": "good_up",
    "C/R": "good_up",
    "ACCEPTANCE RATE": "good_up",
    "DRIVER CANCELATION": "bad_up",
    "SUPPLY UTILIZATION": "context",
    "TAKE RATE (AS % OF GB)": "context",
    # Any metric not listed here renders neutral (no color) \u2014 its
    # business polarity wasn't specified, so nothing is assumed.
}

month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]


def month_label(month):
    year, mo = month.split("-")
    return f"{month_names[int(mo) - 1]}-{year[2:]}"


# Only months whose same month exists one year earlier are offered \u2014
# selecting any other month would never be comparable (spec section 6).
comparable_months = [m for m in all_months_sorted if prior_year_month(m) in all_months_sorted]
default_yoy_month = comparable_months[-1] if comparable_months else all_months_sorted[-1]

yoy_month_options_html = "".join(
    f'<option value="{m}"{" selected" if m == default_yoy_month else ""}>{month_label(m)}</option>'
    for m in comparable_months
)
yoy_city_checkbox_items_html = "".join(
    f'<label><input type="checkbox" class="yoy-city-checkbox" value="{c}" checked> {c}</label>'
    for c in all_cities_sorted
)
yoy_metric_options_html = "".join(
    f'<option value="{m}">{m}</option>' for m in metric_columns
)

yoy_panel_content = f"""
<h2>Year-Over-Year Analysis</h2>
<p class="description">Compares a period against the same calendar month(s) one year earlier \u2014 never a different month, and never a full year against a partial year. Volume metrics are shown as YoY growth (%); rate and conversion metrics are shown as change in percentage points (pp). Choose Full Funnel to see the whole conversion path in order, or pick a single metric.</p>

<div class="chart-controls">
  <div class="control-group">
    <label for="yoy-month-select">Month</label>
    <select id="yoy-month-select">{yoy_month_options_html}</select>
  </div>
  <div class="control-group">
    <label>Cities</label>
    <div class="multiselect" id="yoy-city-multiselect">
      <button type="button" class="multiselect-toggle" id="yoy-city-toggle">All cities</button>
      <div class="multiselect-panel" id="yoy-city-panel" hidden>{yoy_city_checkbox_items_html}</div>
    </div>
  </div>
  <div class="control-group">
    <label for="yoy-view-select">View</label>
    <select id="yoy-view-select">
      <option value="monthly" selected>Monthly</option>
      <option value="ytd">YTD</option>
    </select>
  </div>
  <div class="control-group">
    <label for="yoy-scope-select">Metric Scope</label>
    <select id="yoy-scope-select">
      <option value="one">One Metric</option>
    </select>
  </div>
  <div class="control-group" id="yoy-metric-group" style="display:none;">
    <label for="yoy-metric-select">Metric</label>
    <select id="yoy-metric-select">{yoy_metric_options_html}</select>
  </div>
</div>

<div class="yoy-table-wrap">
  <table class="yoy-table" id="yoy-table">
    <thead id="yoy-table-head"></thead>
    <tbody id="yoy-table-body"></tbody>
  </table>
</div>

<p class="yoy-narrative" id="yoy-narrative"></p>

<p class="yoy-methodology" title="Volumes are shown as YoY growth %. Rates are shown as change in percentage points.">
  <strong>Methodology:</strong> Year-over-year performance compares the selected period with the same calendar period in the prior year. Volume metrics are reported as percentage change, while rates and conversion metrics are reported as percentage-point change. Marketplace-level ratios are recalculated from their underlying aggregated numerators and denominators rather than averaged across cities. Missing periods are not imputed as zero.
</p>
"""

yoy_css = """
  .yoy-table-wrap { overflow-x: auto; border: 1px solid var(--border); border-radius: var(--radius-sm); margin: 8px 0 16px; }
  .yoy-table { width: 100%; border-collapse: collapse; font-size: 13px; }
  .yoy-table th { background: var(--ink); color: #FFFFFF; text-transform: uppercase; font-size: 11px; letter-spacing: 0.4px; padding: 12px 14px; text-align: left; }
  .yoy-table td { padding: 12px 14px; border-bottom: 1px solid var(--border); font-family: var(--mono); font-size: 12.5px; }
  .yoy-table td.yoy-metric-cell { font-family: var(--sans); font-weight: 700; font-size: 13px; }
  .yoy-table tbody tr:last-child td { border-bottom: none; }
  .yoy-table tbody tr:hover { background: var(--canvas); }
  .yoy-change-good { color: var(--ok); font-weight: 700; }
  .yoy-change-bad { color: var(--bad); font-weight: 700; }
  .yoy-change-context { color: var(--warn); font-weight: 700; }
  .yoy-change-neutral { color: var(--ink); font-weight: 700; }
  .yoy-note { color: var(--muted); font-style: italic; font-family: var(--sans); }
  .yoy-narrative { margin: 4px 0 16px; padding: 14px 16px; background: var(--canvas); border-left: 4px solid var(--blue); border-radius: var(--radius-sm); font-size: 13px; line-height: 1.6; color: var(--ink); }
  .yoy-methodology { font-size: 11.5px; color: var(--muted); line-height: 1.5; border-top: 1px solid var(--border); padding-top: 12px; cursor: help; }
"""

yoy_js_template = """
  const yoyFunnelMetrics = __FUNNEL_METRICS__;
  const yoyRatioComponents = __RATIO_COMPONENTS__;
  const yoyRatioMetrics = __RATIO_METRICS__;
  const yoyMonthNames = __MONTH_NAMES__;
  const yoyPolarity = __POLARITY__;

  const yoyMonthSelect = document.getElementById('yoy-month-select');
  const yoyCityToggle = document.getElementById('yoy-city-toggle');
  const yoyCityPanel = document.getElementById('yoy-city-panel');
  const yoyCityCheckboxes = Array.from(document.querySelectorAll('.yoy-city-checkbox'));
  const yoyViewSelect = document.getElementById('yoy-view-select');
  const yoyScopeSelect = document.getElementById('yoy-scope-select');
  const yoyMetricGroup = document.getElementById('yoy-metric-group');
  const yoyMetricSelect = document.getElementById('yoy-metric-select');
  const yoyTableHead = document.getElementById('yoy-table-head');
  const yoyTableBody = document.getElementById('yoy-table-body');
  const yoyNarrativeEl = document.getElementById('yoy-narrative');

  function yoyMonthLabel(month) {
    const parts = month.split('-');
    return yoyMonthNames[parseInt(parts[1], 10) - 1] + '-' + parts[0].slice(2);
  }

  function yoyPriorYearMonth(month) {
    const parts = month.split('-');
    return (parseInt(parts[0], 10) - 1) + '-' + parts[1];
  }

  function yoySelectedCities() {
    return yoyCityCheckboxes.filter(function(cb) { return cb.checked; }).map(function(cb) { return cb.value; });
  }

  function yoyUpdateCityToggleLabel() {
    const selected = yoySelectedCities();
    if (selected.length === 0) {
      yoyCityToggle.textContent = 'No cities selected';
    } else if (selected.length === chartData.cities.length) {
      yoyCityToggle.textContent = 'All cities (' + selected.length + ')';
    } else if (selected.length <= 2) {
      yoyCityToggle.textContent = selected.join(', ');
    } else {
      yoyCityToggle.textContent = selected.length + ' cities selected';
    }
  }

  yoyCityToggle.addEventListener('click', function(event) {
    event.stopPropagation();
    yoyCityPanel.hidden = !yoyCityPanel.hidden;
  });
  document.addEventListener('click', function(event) {
    if (!yoyCityPanel.hidden && !yoyCityPanel.contains(event.target) && event.target !== yoyCityToggle) {
      yoyCityPanel.hidden = true;
    }
  });

  function yoyGetValue(metric, city, month) {
    const idx = chartData.months.indexOf(month);
    if (idx === -1) return null;
    const series = chartData.metrics[metric] && chartData.metrics[metric][city];
    if (!series) return null;
    const value = series[idx];
    return (value === null || value === undefined) ? null : value;
  }

  function yoySumMetric(metric, months, cities) {
    let total = 0;
    let any = false;
    cities.forEach(function(city) {
      months.forEach(function(month) {
        const value = yoyGetValue(metric, city, month);
        if (value !== null) { total += value; any = true; }
      });
    });
    return any ? total : null;
  }

  function yoyAggregateRatio(metric, months, cities) {
    const components = yoyRatioComponents[metric];
    if (components) {
      const denominator = yoySumMetric(components[1], months, cities);
      if (!denominator) return null;
      const numerator = yoySumMetric(components[0], months, cities);
      if (numerator === null) return null;
      return (numerator / denominator) * 100;
    }
    const values = [];
    cities.forEach(function(city) {
      months.forEach(function(month) {
        const value = yoyGetValue(metric, city, month);
        if (value !== null) values.push(value);
      });
    });
    if (values.length === 0) return null;
    return values.reduce(function(a, b) { return a + b; }, 0) / values.length;
  }

  function yoyFormatVolume(value) {
    if (value === null) return '\u2013';
    const abs = Math.abs(value);
    if (abs >= 1000000) return (value / 1000000).toFixed(1) + 'M';
    if (abs >= 1000) return Math.round(value / 1000) + 'K';
    return Math.round(value).toString();
  }

  function yoyFormatRate(value) {
    return value === null ? '\u2013' : value.toFixed(1) + '%';
  }

  function yoyComputeRow(metric, currentMonths, priorMonths, cities) {
    const isRatio = yoyRatioMetrics.indexOf(metric) !== -1;
    const missingPrior = priorMonths.some(function(m) { return chartData.months.indexOf(m) === -1; });
    const missingCurrent = currentMonths.some(function(m) { return chartData.months.indexOf(m) === -1; });

    if (missingPrior || missingCurrent) {
      return {
        metric: metric, isRatio: isRatio,
        priorDisplay: '\u2013', currentDisplay: '\u2013',
        changeDisplay: 'Not comparable \u2014 matching prior-year month is unavailable.',
        change: undefined,
      };
    }

    if (isRatio) {
      const priorValue = yoyAggregateRatio(metric, priorMonths, cities);
      const currentValue = yoyAggregateRatio(metric, currentMonths, cities);
      if (priorValue === null || currentValue === null) {
        return {
          metric: metric, isRatio: isRatio,
          priorDisplay: yoyFormatRate(priorValue), currentDisplay: yoyFormatRate(currentValue),
          changeDisplay: 'Not available \u2014 prior-year value is zero.',
          change: undefined,
        };
      }
      const change = currentValue - priorValue;
      const sign = change >= 0 ? '+' : '\u2212';
      return {
        metric: metric, isRatio: isRatio, change: change,
        priorDisplay: yoyFormatRate(priorValue), currentDisplay: yoyFormatRate(currentValue),
        changeDisplay: sign + Math.abs(change).toFixed(1) + ' pp',
      };
    }

    const priorValue = yoySumMetric(metric, priorMonths, cities);
    const currentValue = yoySumMetric(metric, currentMonths, cities);
    if (!priorValue) {
      return {
        metric: metric, isRatio: isRatio,
        priorDisplay: yoyFormatVolume(priorValue), currentDisplay: yoyFormatVolume(currentValue),
        changeDisplay: 'Not available \u2014 prior-year value is zero.',
        change: undefined,
      };
    }
    const changePct = (currentValue / priorValue - 1) * 100;
    const sign = changePct >= 0 ? '+' : '\u2212';
    return {
      metric: metric, isRatio: isRatio, change: changePct,
      priorDisplay: yoyFormatVolume(priorValue), currentDisplay: yoyFormatVolume(currentValue),
      changeDisplay: sign + Math.abs(changePct).toFixed(1) + '%',
    };
  }

  function yoyYtdMonths(targetMonth) {
    const parts = targetMonth.split('-');
    const year = parts[0];
    const lastMo = parseInt(parts[1], 10);
    const months = [];
    for (let m = 1; m <= lastMo; m++) {
      months.push(year + '-' + String(m).padStart(2, '0'));
    }
    return months;
  }

  function yoyBuildPeriods() {
    const targetMonth = yoyMonthSelect.value;
    const view = yoyViewSelect.value;
    if (view === 'ytd') {
      const currentMonths = yoyYtdMonths(targetMonth);
      const priorMonths = currentMonths.map(yoyPriorYearMonth);
      return {
        currentMonths: currentMonths, priorMonths: priorMonths,
        priorLabel: yoyMonthLabel(priorMonths[0]) + ' \u2013 ' + yoyMonthLabel(priorMonths[priorMonths.length - 1]),
        currentLabel: yoyMonthLabel(currentMonths[0]) + ' \u2013 ' + yoyMonthLabel(currentMonths[currentMonths.length - 1]),
      };
    }
    const priorMonth = yoyPriorYearMonth(targetMonth);
    return {
      currentMonths: [targetMonth], priorMonths: [priorMonth],
      priorLabel: yoyMonthLabel(priorMonth), currentLabel: yoyMonthLabel(targetMonth),
    };
  }

  function yoyPolarityClass(metric, change) {
    if (change === undefined) return 'yoy-change-neutral';
    const polarity = yoyPolarity[metric];
    if (polarity === 'good_up') return change >= 0 ? 'yoy-change-good' : 'yoy-change-bad';
    if (polarity === 'bad_up') return change >= 0 ? 'yoy-change-bad' : 'yoy-change-good';
    if (polarity === 'context') return 'yoy-change-context';
    return 'yoy-change-neutral';
  }

  function yoyRenderTable(periods, metrics) {
    yoyTableHead.innerHTML =
      '<tr><th>Metric</th><th>' + periods.priorLabel + '</th><th>' + periods.currentLabel + '</th><th>YoY Change</th></tr>';

    const cities = yoySelectedCities();
    const rows = metrics.map(function(metric) {
      return yoyComputeRow(metric, periods.currentMonths, periods.priorMonths, cities);
    });

    yoyTableBody.innerHTML = rows.map(function(row) {
      const isNote = row.change === undefined;
      const changeClass = isNote ? 'yoy-note' : yoyPolarityClass(row.metric, row.change);
      return (
        '<tr>' +
        '<td class="yoy-metric-cell">' + row.metric + '</td>' +
        '<td>' + row.priorDisplay + '</td>' +
        '<td>' + row.currentDisplay + '</td>' +
        '<td class="' + changeClass + '">' + row.changeDisplay + '</td>' +
        '</tr>'
      );
    }).join('');

    return rows;
  }

  function yoyDirectionWord(change) {
    return change >= 0 ? 'increased' : 'declined';
  }

  function yoyDirectionNoun(change) {
    return change >= 0 ? 'increase' : 'decline';
  }

  function yoyBuildNarrative(rows) {
    const byName = {};
    rows.forEach(function(row) { byName[row.metric] = row; });
    const needed = ['SHOPPING SESSIONS', 'RS/SS', 'REQUESTING SESSIONS', 'C/RS', 'TRIPS', 'C/SS'];
    const hasAll = needed.every(function(m) { return byName[m] && byName[m].change !== undefined; });
    if (!hasAll) {
      return 'Not enough comparable data is available to build a narrative for this selection.';
    }

    const ss = byName['SHOPPING SESSIONS'];
    const rsss = byName['RS/SS'];
    const rs = byName['REQUESTING SESSIONS'];
    const crs = byName['C/RS'];
    const trips = byName['TRIPS'];
    const css = byName['C/SS'];

    const intentMag = Math.abs(rsss.change);
    const fulfillmentMag = Math.abs(crs.change);
    let driver;
    if (intentMag > fulfillmentMag * 1.2) {
      driver = 'appears to be primarily driven by the Intent stage (RS/SS) rather than the Fulfillment stage (C/RS)';
    } else if (fulfillmentMag > intentMag * 1.2) {
      driver = 'appears to be primarily driven by the Fulfillment stage (C/RS) rather than the Intent stage (RS/SS)';
    } else {
      driver = 'may be associated with both the Intent (RS/SS) and Fulfillment (C/RS) stages fairly evenly';
    }

    return (
      'Shopping Sessions ' + yoyDirectionWord(ss.change) + ' by ' + Math.abs(ss.change).toFixed(1) + '% year over year. ' +
      'RS/SS ' + yoyDirectionWord(rsss.change) + ' by ' + Math.abs(rsss.change).toFixed(1) + ' percentage points, which is associated with a ' +
      Math.abs(rs.change).toFixed(1) + '% ' + yoyDirectionNoun(rs.change) + ' in Requesting Sessions. ' +
      'C/RS ' + yoyDirectionWord(crs.change) + ' by ' + Math.abs(crs.change).toFixed(1) + ' percentage points. ' +
      'This suggests the overall change in conversion ' + driver + '. ' +
      'As a result, Trips ' + yoyDirectionWord(trips.change) + ' by ' + Math.abs(trips.change).toFixed(1) + '%, ' +
      'and overall C/SS ' + yoyDirectionWord(css.change) + ' by ' + Math.abs(css.change).toFixed(1) + ' percentage points.'
    );
  }

  function yoyRender() {
    const periods = yoyBuildPeriods();
    const scope = yoyScopeSelect.value;
    const metrics = scope === 'funnel' ? yoyFunnelMetrics : [yoyMetricSelect.value];
    const rows = yoyRenderTable(periods, metrics);

    if (scope === 'funnel') {
      yoyNarrativeEl.style.display = '';
      yoyNarrativeEl.textContent = yoyBuildNarrative(rows);
    } else {
      yoyNarrativeEl.style.display = 'none';
    }
  }

  yoyScopeSelect.addEventListener('change', function() {
    yoyMetricGroup.style.display = yoyScopeSelect.value === 'one' ? '' : 'none';
    yoyRender();
  });
  yoyMonthSelect.addEventListener('change', yoyRender);
  yoyViewSelect.addEventListener('change', yoyRender);
  yoyMetricSelect.addEventListener('change', yoyRender);
  yoyCityCheckboxes.forEach(function(cb) {
    cb.addEventListener('change', function() {
      yoyUpdateCityToggleLabel();
      yoyRender();
    });
  });

  yoyUpdateCityToggleLabel();
  yoyRender();
"""

yoy_js = (
    yoy_js_template
    .replace("__RATIO_COMPONENTS__", json.dumps(ratio_components))
    .replace("__RATIO_METRICS__", json.dumps(sorted(ratio_metrics)))
    .replace("__MONTH_NAMES__", json.dumps(month_names))
    .replace("__POLARITY__", json.dumps(yoy_polarity))
)

# ============================================
# Insert into dashboard.html (idempotent: replaces prior content between
# the same markers if this cell is re-run)
# ============================================
dashboard_path = Path("dashboard.html")
current_html = dashboard_path.read_text(encoding="utf-8")

content_start = "<!-- YOY_CONTENT_START -->"
content_end = "<!-- YOY_CONTENT_END -->"
before_content = current_html.split(content_start, 1)[0] + content_start
after_content = current_html.split(content_end, 1)[1]
current_html = before_content + "\n" + yoy_panel_content + "\n" + content_end + after_content

style_close = "</style>"
current_html = current_html.replace(style_close, yoy_css + style_close, 1)

js_start = "// YOY_JS_START"
js_end = "// YOY_JS_END"
before_js = current_html.split(js_start, 1)[0] + js_start
after_js = current_html.split(js_end, 1)[1]
current_html = before_js + "\n" + yoy_js + "\n  " + js_end + after_js

dashboard_path.write_text(current_html, encoding="utf-8")
print("Year-over-Year Analysis panel added to dashboard.html")
print("Comparable months available:", comparable_months)


Year-over-Year Analysis panel added to dashboard.html
Comparable months available: ['2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-06', '2023-07', '2023-08', '2023-09']


## 7. Data Quality — Completeness Calculations

Calculate completion percentage, null count, and city/month coverage directly from the dataset.

In [108]:
total_rows = len(df_quality)
total_cols = len(df_quality.columns)
total_cells = total_rows * total_cols

null_count = int(df_quality.isna().sum().sum())
completeness_pct = round((1 - null_count / total_cells) * 100, 1)
completeness_pct_label = f"{completeness_pct:.0f}" if completeness_pct == int(completeness_pct) else f"{completeness_pct}"

n_cities = df_quality['CITY'].nunique()
n_months = df_quality['MONTH'].nunique()

if null_count == 0:
    summary_text = (
        f"Total columns: {total_cols} \u00b7 {total_rows} records \u00b7 "
        f"{n_cities} cities \u00b7 {n_months} months \u00b7 no missing values"
    )
else:
    summary_text = (
        f"Total columns: {total_cols} \u00b7 {total_rows} records \u00b7 "
        f"{n_cities} cities \u00b7 {n_months} months \u00b7 {null_count} missing values"
    )

completeness_pct, null_count, summary_text

(100.0,
 0,
 'Total columns: 22 · 168 records · 8 cities · 21 months · no missing values')

## 8. Build the Data Quality Card (`data_quality.html`)

Generate a compact, centered Uber-blue hero card using the values calculated above.

In [109]:
# ============================================
# CONFIGURE HERE: colors and label
# ============================================
uber_blue = '#276EF1'
card_label = 'Data Completeness'

data_quality_html = f'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Data Quality</title>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; font-family: 'Segoe UI', Arial, sans-serif; }}
  body {{
    background: #f3f4f6;
    padding: 24px;
    min-height: 100vh;
    display: flex;
    align-items: center;
    justify-content: center;
  }}

  .hero-card {{
    background: {uber_blue};
    color: #FFFFFF;
    border-radius: 16px;
    padding: 28px 40px;
    display: flex;
    flex-direction: column;
    align-items: center;
    text-align: center;
    gap: 4px;
    box-shadow: 0 4px 20px rgba(39,110,241,0.25);
    max-width: 320px;
  }}
  .hero-card .check {{ font-size: 32px; line-height: 1; margin-bottom: 4px; }}
  .hero-card .hero-number {{ font-size: 44px; font-weight: 700; line-height: 1; }}
  .hero-card .hero-label {{ font-size: 14px; font-weight: 600; opacity: 0.95; }}
  .hero-card .hero-sub {{ font-size: 11px; opacity: 0.8; }}
</style>
</head>
<body>

  <div class="hero-card">
    <div class="check">\u2705</div>
    <div class="hero-number">{completeness_pct_label}%</div>
    <div class="hero-label">{card_label}</div>
    <div class="hero-sub">{summary_text}</div>
  </div>

</body>
</html>
'''

with open('data_quality.html', 'w', encoding='utf-8') as f:
    f.write(data_quality_html)

print("data_quality.html generated")

data_quality.html generated


## 9. Data Quality — Per-Column Breakdown

Calculate completeness, null count, and status for each column. These values populate the dropdown in `data_quality.html`.

In [110]:
# ============================================
# CONFIGURE HERE: short description per column
# and the OK / warning / critical thresholds
# NOTE: "NPI" description below is a placeholder — confirm the real definition
# ============================================
column_descriptions = {
    "MONTH": "Reporting month for the record (YYYY-MM).",
    "CITY": "Market / city where the metrics were recorded.",
    "TRIPS": "Number of completed trips.",
    "GROSS BOOKINGS (GB)": "Total gross bookings value generated.",
    "SHOPPING SESSIONS": "App sessions where a rider browsed, not necessarily requesting a ride.",
    "RIDERS WITH SESSIONS": "Unique riders who opened a session.",
    "REQUESTING SESSIONS": "Sessions where the rider went on to request a ride.",
    "REQUESTS": "Total ride requests made.",
    "ACTIVE DRIVERS": "Unique drivers who were online during the period.",
    "SUPPLY HOURS": "Total hours drivers spent online and available.",
    "C/R": "Completion rate: completed trips over requests.",
    "C/RS": "Completion rate: completed trips over requesting sessions.",
    "RS/SS": "Conversion rate: requesting sessions over shopping sessions.",
    "C/SS": "Completion rate: completed trips over shopping sessions.",
    "ACCEPTANCE RATE": "Share of requests accepted by drivers.",
    "DRIVER CANCELATION": "Share of trips cancelled by drivers.",
    "SUPPLY UTILIZATION": "Share of available driver supply hours actually used.",
    "NPI": "Placeholder \u2014 confirm real definition.",
    "DRIVER PAYMENTS (AS % OF GB)": "Driver earnings as a percentage of gross bookings.",
    "RIDER INCENTIVES AND PROMOTIONS (AS % OF GB)": "Rider-side incentives/promos as a percentage of gross bookings.",
    "DRIVER INCENTIVES AND PROMOTIONS (AS % OF GB)": "Driver-side incentives/promos as a percentage of gross bookings.",
    "TAKE RATE (AS % OF GB)": "Platform\'s take rate as a percentage of gross bookings.",
}

ok_threshold = 98      # >= this % -> OK
warn_threshold = 90    # >= this % (and below ok_threshold) -> warning

column_stats = []
for col in df_quality.columns:
    nulls = int(df_quality[col].isna().sum())
    pct = round((1 - nulls / total_rows) * 100, 1)
    pct_label = f"{pct:.0f}" if pct == int(pct) else f"{pct}"
    if pct >= ok_threshold:
        status = "ok"
    elif pct >= warn_threshold:
        status = "warn"
    else:
        status = "bad"
    column_stats.append({
        "name": col,
        "description": column_descriptions.get(col, "No description available."),
        "completeness": pct_label,
        "nulls": nulls,
        "total": total_rows,
        "status": status,
    })

column_stats

[{'name': 'MONTH',
  'description': 'Reporting month for the record (YYYY-MM).',
  'completeness': '100',
  'nulls': 0,
  'total': 168,
  'status': 'ok'},
 {'name': 'CITY',
  'description': 'Market / city where the metrics were recorded.',
  'completeness': '100',
  'nulls': 0,
  'total': 168,
  'status': 'ok'},
 {'name': 'TRIPS',
  'description': 'Number of completed trips.',
  'completeness': '100',
  'nulls': 0,
  'total': 168,
  'status': 'ok'},
 {'name': 'GROSS BOOKINGS (GB)',
  'description': 'Total gross bookings value generated.',
  'completeness': '100',
  'nulls': 0,
  'total': 168,
  'status': 'ok'},
 {'name': 'SHOPPING SESSIONS',
  'description': 'App sessions where a rider browsed, not necessarily requesting a ride.',
  'completeness': '100',
  'nulls': 0,
  'total': 168,
  'status': 'ok'},
 {'name': 'RIDERS WITH SESSIONS',
  'description': 'Unique riders who opened a session.',
  'completeness': '100',
  'nulls': 0,
  'total': 168,
  'status': 'ok'},
 {'name': 'REQUESTING

## 10. Build the Data Nulls Section (`data_quality.html`)

Title, description, KPI card, and a dropdown to drill into each column \u2014 all populated from `column_stats` above, so editing the CSV updates this automatically.

In [111]:
# ============================================
# CONFIGURE HERE: titles, copy and colors
# ============================================
uber_blue = "#2171EC"
uber_blue_dark = "#1257B8"
uber_blue_tint = "#E9F0FE"
uber_ink = "#121212"
uber_muted = "#6B6F76"
uber_faint = "#ADADAD"
uber_canvas = "#F1F2F4"
uber_surface = "#FFFFFF"
uber_border = "#DEE1E6"
uber_green = "#1F8B4C"
uber_green_bg = "#E6F4EC"
uber_yellow = "#B7791F"
uber_yellow_bg = "#FCF6EA"
uber_red = "#C4291C"
uber_red_bg = "#FBEAE8"

page_title = "DATA NULLS"
page_description = (
    "This visualization shows the number and percentage of missing values "
    "in each column, making it easy to identify data gaps and prioritize "
    "fields that may require cleaning. Select a column below to explore "
    "its missing-value details."
)

status_colors = {"ok": uber_green, "warn": uber_yellow, "bad": uber_red}
status_labels = {"ok": "OK", "warn": "Review", "bad": "Critical"}

column_stats_json = json.dumps(column_stats)

data_quality_html = f'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Data Quality</title>
<style>
  :root {{
    --canvas: {uber_canvas};
    --surface: {uber_surface};
    --border: {uber_border};
    --ink: {uber_ink};
    --muted: {uber_muted};
    --faint: {uber_faint};
    --blue: {uber_blue};
    --blue-dark: {uber_blue_dark};
    --blue-tint: {uber_blue_tint};
    --ok: {uber_green};
    --ok-bg: {uber_green_bg};
    --warn: {uber_yellow};
    --warn-bg: {uber_yellow_bg};
    --bad: {uber_red};
    --bad-bg: {uber_red_bg};
    --radius: 10px;
    --radius-sm: 6px;
    --sans: -apple-system, BlinkMacSystemFont, \'Segoe UI\', Roboto, Helvetica, Arial, sans-serif;
    --mono: ui-monospace, SFMono-Regular, \'Roboto Mono\', Menlo, Consolas, monospace;
  }}

  * {{ box-sizing: border-box; margin: 0; padding: 0; font-family: var(--sans); }}
  body {{ background: var(--canvas); color: var(--ink); padding: 40px 32px; }}

  #data-nulls, #month-coverage, #data-type-check, #funnel-ratio-validation {{ scroll-margin-top: 24px; }}

  .wrap {{ max-width: 1400px; margin: 0 auto; }}

  h2 {{
    position: relative;
    color: #FFFFFF;
    background: var(--ink);
    margin: -28px -32px 22px;
    padding: 22px 32px 24px;
    text-align: center;
    text-transform: uppercase;
    font-size: 21px;
    font-weight: 800;
    letter-spacing: 0.08em;
    border-radius: var(--radius) var(--radius) 0 0;
  }}
  h2::before {{
    content: "";
    position: absolute;
    left: 50%;
    bottom: 12px;
    transform: translateX(-50%);
    width: 40px;
    height: 3px;
    background: var(--blue);
    border-radius: 2px;
  }}
  .description {{ color: var(--muted); font-size: 13px; margin: 0 0 22px; line-height: 1.5; text-align: left; }}

  .nulls-grid {{ display: flex; flex-direction: column; gap: 28px; margin-top: 4px; }}
  .nulls-col {{ display: flex; flex-direction: column; }}
  .nulls-kicker {{ display: flex; align-items: center; gap: 8px; font-size: 13px; font-weight: 700; text-transform: uppercase; letter-spacing: 0.08em; color: var(--ink); margin-bottom: 10px; }}
  .nulls-kicker::before {{ content: ""; width: 6px; height: 6px; border-radius: 50%; background: var(--blue); flex-shrink: 0; }}

  .hero-card {{
    background: linear-gradient(135deg, var(--blue) 0%, var(--blue-dark) 100%);
    color: #FFFFFF;
    border-radius: var(--radius);
    padding: 28px 40px;
    display: flex;
    flex-direction: column;
    align-items: center;
    text-align: center;
    gap: 10px;
    max-width: 640px;
    width: 100%;
    margin: 0 auto;
    box-shadow: 0 8px 24px rgba(33, 113, 236, 0.25);
  }}
  .hero-card .hero-number {{ font-family: var(--mono); font-size: 48px; font-weight: 700; line-height: 1; color: #FFFFFF; }}
  .hero-card .hero-label {{ font-size: 12px; font-weight: 700; text-transform: uppercase; letter-spacing: 0.08em; color: #FFFFFF; opacity: 0.9; }}
  .hero-bar {{ height: 4px; width: 160px; background: rgba(255,255,255,0.3); border-radius: 2px; overflow: hidden; margin: 4px 0; }}
  .hero-bar-fill {{ height: 100%; background: #FFFFFF; }}
  .hero-card .hero-sub {{ font-size: 11.5px; color: #FFFFFF; opacity: 0.85; line-height: 1.6; max-width: 520px; }}

  .section-card {{ background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); padding: 28px 32px; box-shadow: 0 1px 2px rgba(18,18,18,0.04); }}
  .section-card label {{ font-size: 11px; text-transform: uppercase; letter-spacing: 0.04em; font-weight: 700; color: var(--muted); display: block; margin-bottom: 8px; }}
  select {{
    width: 100%;
    padding: 9px 34px 9px 12px;
    border-radius: var(--radius-sm);
    border: 1px solid var(--border);
    font-size: 13px;
    font-family: var(--sans);
    margin-bottom: 18px;
    background: var(--surface);
    color: var(--ink);
  }}

  .col-name {{ font-size: 16px; font-weight: 700; color: var(--ink); margin-bottom: 10px; }}
  .metric-row {{ font-size: 13px; color: var(--muted); margin-bottom: 8px; }}
  .metric-label {{ color: var(--muted); }}
  #col-pct, #col-nulls, #col-total {{ font-family: var(--mono); font-weight: 700; color: var(--ink); }}
  .badge {{ padding: 3px 10px 3px 8px; border-radius: 4px; font-size: 11px; font-weight: 700; text-transform: uppercase; letter-spacing: 0.03em; border-left: 3px solid currentColor; }}

  .table-scroll {{ overflow-x: auto; border-radius: var(--radius-sm); border: 1px solid var(--border); }}
  .coverage-table {{ width: 100%; border-collapse: collapse; font-size: 13px; }}
  .coverage-table th, .coverage-table td {{ padding: 12px 14px; white-space: nowrap; }}
  .coverage-table thead th {{
    text-align: center;
    color: #FFFFFF;
    font-weight: 700;
    font-size: 11px;
    text-transform: uppercase;
    letter-spacing: 0.4px;
    background: var(--ink);
    border-bottom: 1px solid var(--ink);
  }}
  .coverage-table thead th:first-child {{ text-align: left; }}
  .coverage-table tbody tr {{ border-bottom: 1px solid var(--border); }}
  .coverage-table tbody tr:last-child {{ border-bottom: none; }}
  .coverage-table tbody tr:hover {{ background: var(--blue-tint); }}
  .city-cell {{ font-weight: 600; color: var(--ink); text-align: left; }}
  .num-cell {{ text-align: center; color: var(--ink); font-family: var(--mono); }}
  .muted-cell {{ text-align: center; color: var(--faint); font-size: 12px; font-family: var(--mono); }}
  .cov-badge {{ display: inline-block; padding: 4px 10px 4px 8px; border-radius: 4px; font-weight: 700; font-family: var(--mono); font-size: 12px; }}
  .cov-badge-ok {{ background: var(--blue); color: #FFFFFF; border-left: 3px solid var(--ink); }}
  .cov-badge-warn {{ background: var(--warn-bg); color: var(--warn); border-left: 3px solid var(--warn); }}
  .cov-badge-bad {{ background: var(--bad-bg); color: var(--bad); border-left: 3px solid var(--bad); }}
</style>
</head>
<body>
  <div class="wrap">
    <div class="section-card" id="data-nulls">
      <h2>{page_title}</h2>
      <p class="description">{page_description}</p>

      <div class="nulls-grid">
        <div class="nulls-col">
          <div class="nulls-kicker">Overall Results</div>
          <div class="hero-card">
            <div class="hero-number">{completeness_pct_label}%</div>
            <div class="hero-label">Data Completeness</div>
            <div class="hero-bar"><div class="hero-bar-fill" style="width:{completeness_pct_label}%"></div></div>
            <div class="hero-sub">{summary_text}</div>
          </div>
        </div>

        <div class="nulls-col">
          <div class="nulls-kicker">Metric Details</div>
          <div class="detail-panel">
            <label for="column-select">Choose a column</label>
            <select id="column-select" onchange="renderColumn(this.value)"></select>

            <div class="col-name" id="col-name"></div>
            <div class="metric-row"><span class="metric-label">Completion rate:</span> <span id="col-pct"></span> <span class="badge" id="col-badge"></span></div>
            <div class="metric-row"><span class="metric-label">Total nulls:</span> <span id="col-nulls"></span></div>
            <div class="metric-row"><span class="metric-label">Total data:</span> <span id="col-total"></span></div>
          </div>
        </div>
      </div>
    </div>

<script>
  const columnStats = {column_stats_json};
  const statusColors = {{ ok: "{uber_green}", warn: "{uber_yellow}", bad: "{uber_red}" }};
  const statusLabels = {{ ok: "OK", warn: "Review", bad: "Critical" }};

  const select = document.getElementById(\'column-select\');
  columnStats.forEach(function(col) {{
    const opt = document.createElement(\'option\');
    opt.value = col.name;
    opt.textContent = col.name;
    select.appendChild(opt);
  }});

  function renderColumn(name) {{
    const col = columnStats.find(function(c) {{ return c.name === name; }});
    if (!col) return;
    document.getElementById(\'col-name\').textContent = col.name;
    document.getElementById(\'col-pct\').textContent = col.completeness + "%";
    document.getElementById(\'col-pct\').style.color = statusColors[col.status];
    document.getElementById(\'col-nulls\').textContent = col.nulls;
    document.getElementById(\'col-total\').textContent = col.total;
    const badge = document.getElementById(\'col-badge\');
    badge.textContent = statusLabels[col.status];
    badge.style.background = statusColors[col.status] + "22";
    badge.style.color = statusColors[col.status];
  }}

  renderColumn(columnStats[0].name);
</script>
</body>
</html>
'''

with open("data_quality.html", "w", encoding="utf-8") as f:
    f.write(data_quality_html)

print("data_quality.html generated with", len(column_stats), "columns")


data_quality.html generated with 22 columns


## 11. Data Quality — Month Coverage by City

For each city, calculate how many months it *should* contain based on the full dataset date range, how many it actually contains, and any missing or duplicated months. These values populate the Month Coverage table in `data_quality.html`.

In [112]:
from collections import Counter

all_months = sorted(df_quality['MONTH'].unique())
expected_months = len(all_months)
first_month = all_months[0]
last_month = all_months[-1]

city_coverage = []
for city, g in df_quality.groupby('CITY'):
    months_list = g['MONTH'].tolist()
    unique_months = sorted(set(months_list))
    missing = sorted(set(all_months) - set(unique_months))
    dupes = sorted([m for m, c in Counter(months_list).items() if c > 1])
    coverage_pct = round(len(unique_months) / expected_months * 100, 1)
    coverage_label = f"{coverage_pct:.0f}" if coverage_pct == int(coverage_pct) else f"{coverage_pct}"

    if coverage_pct >= ok_threshold:
        cov_status = "ok"
    elif coverage_pct >= warn_threshold:
        cov_status = "warn"
    else:
        cov_status = "bad"

    city_coverage.append({
        "city": city,
        "expected_months": expected_months,
        "rows_found": len(g),
        "unique_months": len(unique_months),
        "coverage": coverage_label,
        "missing_months": ", ".join(missing) if missing else "None",
        "duplicate_months": ", ".join(dupes) if dupes else "None",
        "status": cov_status,
    })

city_coverage

[{'city': 'Azteca',
  'expected_months': 21,
  'rows_found': 21,
  'unique_months': 21,
  'coverage': '100',
  'missing_months': 'None',
  'duplicate_months': 'None',
  'status': 'ok'},
 {'city': 'Lusail',
  'expected_months': 21,
  'rows_found': 21,
  'unique_months': 21,
  'coverage': '100',
  'missing_months': 'None',
  'duplicate_months': 'None',
  'status': 'ok'},
 {'city': 'Maracanã',
  'expected_months': 21,
  'rows_found': 21,
  'unique_months': 21,
  'coverage': '100',
  'missing_months': 'None',
  'duplicate_months': 'None',
  'status': 'ok'},
 {'city': 'Old Trafford',
  'expected_months': 21,
  'rows_found': 21,
  'unique_months': 21,
  'coverage': '100',
  'missing_months': 'None',
  'duplicate_months': 'None',
  'status': 'ok'},
 {'city': 'Rose Bowl',
  'expected_months': 21,
  'rows_found': 21,
  'unique_months': 21,
  'coverage': '100',
  'missing_months': 'None',
  'duplicate_months': 'None',
  'status': 'ok'},
 {'city': 'San Siro',
  'expected_months': 21,
  'rows_foun

## 12. Add the Month Coverage Table to `data_quality.html`

Append a second section card below the existing section using `city_coverage`, so it updates automatically when the CSV changes.

In [113]:
# ============================================
# CONFIGURE HERE: table title and copy
# ============================================
coverage_title = "MONTH COVERAGE"
coverage_description = (
    f"Checks that every city has a row for each month between {first_month} and {last_month} "
    "(the full date range in the dataset), and flags any missing or duplicated months."
)

city_coverage_json = json.dumps(city_coverage)

coverage_rows_html = ""
for row in city_coverage:
    badge_class = f"cov-badge cov-badge-{row['status']}"
    coverage_rows_html += f'''
        <tr>
          <td class="city-cell">{row["city"]}</td>
          <td class="num-cell">{row["expected_months"]}</td>
          <td class="num-cell">{row["rows_found"]}</td>
          <td class="num-cell">{row["unique_months"]}</td>
          <td class="num-cell"><span class="{badge_class}">{row["coverage"]}%</span></td>
          <td class="muted-cell">{row["missing_months"]}</td>
          <td class="muted-cell">{row["duplicate_months"]}</td>
        </tr>'''

coverage_section_html = f'''
    <div class="section-card" id="month-coverage" style="margin-top:24px;">
      <h2>{coverage_title}</h2>
      <p class="description">{coverage_description}</p>
      <div class="table-scroll">
      <table class="coverage-table">
        <thead>
          <tr>
            <th>City</th>
            <th>Expected Months</th>
            <th>Rows Found</th>
            <th>Unique Months</th>
            <th>Coverage %</th>
            <th>Missing Months</th>
            <th>Duplicate Months</th>
          </tr>
        </thead>
        <tbody>{coverage_rows_html}
        </tbody>
      </table>
      </div>
    </div>'''

# Insert the new section right after the Data Nulls card, before the closing
# script tag, without touching anything already built above. All coverage-table
# CSS classes already live in the stylesheet generated in the previous cell.
data_quality_html = data_quality_html.replace(
    "    </div>\n\n<script>",
    "    </div>\n" + coverage_section_html + "\n\n<script>",
)

with open("data_quality.html", "w", encoding="utf-8") as f:
    f.write(data_quality_html)

print("Month Coverage section added for", len(city_coverage), "cities")


Month Coverage section added for 8 cities


## 13. Data Quality — how pandas actually reads each column

This section reports the real pandas dtype, a representative Python scalar type, an example value, null count, and unique values. It does **not** use Excel's visual formats such as General, Number, or Percentage.

In [114]:
def python_scalar_type(series):
    """Returns the Python/NumPy type name of the first non-null value."""
    non_null = series.dropna()
    if non_null.empty:
        return "No non-null values"
    return type(non_null.iloc[0]).__name__


def dtype_family(series):
    """Groups pandas dtypes into business-friendly labels."""
    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"
    if pd.api.types.is_bool_dtype(series):
        return "boolean"
    if pd.api.types.is_integer_dtype(series):
        return "integer"
    if pd.api.types.is_float_dtype(series):
        return "float"
    if pd.api.types.is_numeric_dtype(series):
        return "numeric"
    if pd.api.types.is_string_dtype(series) or series.dtype == "object":
        return "string/object"
    return str(series.dtype)


dtype_report = []
for col in df_quality.columns:
    non_null = df_quality[col].dropna()
    dtype_report.append({
        "COLUMN": col,
        "PANDAS_DTYPE": str(df_quality[col].dtype),
        "DTYPE_FAMILY": dtype_family(df_quality[col]),
        "PYTHON_TYPE_EXAMPLE": python_scalar_type(df_quality[col]),
        "EXAMPLE_VALUE": non_null.iloc[0] if not non_null.empty else None,
        "NULLS": int(df_quality[col].isna().sum()),
        "UNIQUE_VALUES": int(df_quality[col].nunique(dropna=True)),
    })

dtype_report = pd.DataFrame(dtype_report)
dtype_report

,COLUMN,PANDAS_DTYPE,DTYPE_FAMILY,PYTHON_TYPE_EXAMPLE,EXAMPLE_VALUE,NULLS,UNIQUE_VALUES
0,MONTH,str,string/object,str,2022-01,0,21
1,CITY,str,string/object,str,Vila Belmiro,0,8
2,TRIPS,str,string/object,str,"3,005,706",0,168
3,GROSS BOOKINGS (GB),str,string/object,str,"56,529,569",0,168
4,SHOPPING SESSIONS,str,string/object,str,"5,283,006",0,168
5,RIDERS WITH SESSIONS,str,string/object,str,"711,432",0,168
6,REQUESTING SESSIONS,str,string/object,str,"3,257,826",0,168
7,REQUESTS,str,string/object,str,"3,844,423",0,168
8,ACTIVE DRIVERS,str,string/object,str,"18,795",0,168
9,SUPPLY HOURS,str,string/object,str,"1,564,816",0,168


## Data Type Check

Expected data types are defined manually and compared against the current pandas dtype.


In [115]:
# ============================================================
# 18. DATA TYPE CHECK
# METRICS | CURRENT DATA | EXPECTED DATA | CORRECT
# ============================================================

from pathlib import Path
import html
import pandas as pd


# ============================================================
# 1. Expected data types defined manually
# ============================================================

expected_types = {
    "MONTH": "date",
    "CITY": "str",

    "TRIPS": "int",
    "GROSS BOOKINGS (GB)": "int",
    "SHOPPING SESSIONS": "int",
    "RIDERS WITH SESSIONS": "int",
    "REQUESTING SESSIONS": "int",
    "REQUESTS": "int",
    "ACTIVE DRIVERS": "int",
    "SUPPLY HOURS": "int",

    "C/R": "float",
    "C/RS": "float",
    "RS/SS": "float",
    "C/SS": "float",
    "ACCEPTANCE RATE": "float",
    "DRIVER CANCELATION": "float",
    "SUPPLY UTILIZATION": "float",

    "NPI": "float64",

    "DRIVER PAYMENTS (AS % OF GB)": "float",
    "RIDER INCENTIVES AND PROMOTIONS (AS % OF GB)": "float",
    "DRIVER INCENTIVES AND PROMOTIONS (AS % OF GB)": "float",
    "TAKE RATE (AS % OF GB)": "float",
}


# ============================================================
# 2. Build the four-column validation table
# ============================================================

data_type_table = pd.DataFrame({
    "METRICS": df_quality.columns,
    "CURRENT DATA": [
        str(df_quality[column].dtype)
        for column in df_quality.columns
    ],
    "EXPECTED DATA": [
        expected_types.get(column, "")
        for column in df_quality.columns
    ],
})

data_type_table["CORRECT"] = (
    data_type_table["CURRENT DATA"]
    == data_type_table["EXPECTED DATA"]
).map({
    True: "Yes",
    False: "No",
})


# ============================================================
# 3. Build HTML rows
# ============================================================

data_type_rows_html = ""

for _, row in data_type_table.iterrows():

    correct_class = (
        "dtype-correct"
        if row["CORRECT"] == "Yes"
        else "dtype-incorrect"
    )

    data_type_rows_html += f"""
    <tr>
        <td class="dtype-metric">
            {html.escape(str(row["METRICS"]))}
        </td>

        <td>
            {html.escape(str(row["CURRENT DATA"]))}
        </td>

        <td>
            {html.escape(str(row["EXPECTED DATA"]))}
        </td>

        <td>
            <span class="dtype-badge {correct_class}">
                {html.escape(str(row["CORRECT"]))}
            </span>
        </td>
    </tr>
    """


# ============================================================
# 4. Build Data Type Check HTML section
# ============================================================

data_type_section_html = f"""
<!-- DATA_TYPE_CHECK_START -->

<style>
    .dtype-section {{
        max-width: 1400px;
        margin: 24px auto 0;
        background: var(--surface);
        border: 1px solid var(--border);
        border-radius: var(--radius);
        box-shadow: 0 1px 2px rgba(18, 18, 18, 0.04);
        padding: 28px 32px;
        color: var(--ink);
        font-family: var(--sans);
    }}

    .dtype-section h2 {{
        text-align: center;
        margin: -28px -32px 22px;
        font-size: 21px;
    }}

    .dtype-description {{
        margin: 0 0 22px 14px;
        text-align: left;
        color: var(--muted);
        font-size: 13px;
        line-height: 1.5;
    }}

    .dtype-table-wrapper {{
        overflow-x: auto;
        border: 1px solid var(--border);
        border-radius: var(--radius-sm);
    }}

    .dtype-table {{
        width: 100%;
        border-collapse: collapse;
        min-width: 760px;
    }}

    .dtype-table th {{
        background: var(--ink);
        color: #FFFFFF;
        text-transform: uppercase;
        font-size: 11px;
        letter-spacing: 0.45px;
        padding: 14px 16px;
        text-align: left;
        border-bottom: 1px solid var(--ink);
    }}

    .dtype-table td {{
        padding: 14px 16px;
        border-bottom: 1px solid var(--border);
        font-size: 13px;
    }}

    .dtype-table td:nth-child(2),
    .dtype-table td:nth-child(3) {{
        font-family: var(--mono);
        font-size: 12.5px;
        color: var(--ink);
    }}

    .dtype-metric {{
        font-weight: 700;
    }}

    .dtype-badge {{
        display: inline-block;
        padding: 4px 10px 4px 8px;
        border-radius: 4px;
        font-size: 11px;
        font-weight: 700;
        text-transform: uppercase;
        letter-spacing: 0.03em;
    }}

    .dtype-correct {{
        background: var(--ok-bg);
        color: var(--ok);
        border-left: 3px solid var(--ok);
    }}

    .dtype-incorrect {{
        background: var(--bad-bg);
        color: var(--bad);
        border-left: 3px solid var(--bad);
    }}
</style>


<section class="dtype-section" id="data-type-check">

    <h2>
        DATA TYPE CHECK
    </h2>

    <p class="dtype-description">
        Compares how pandas currently reads each dataset column
        against the expected data type defined manually for the analysis.
    </p>

    <div class="dtype-table-wrapper">

        <table class="dtype-table">

            <thead>
                <tr>
                    <th>Metrics</th>
                    <th>Current Data</th>
                    <th>Expected Data</th>
                    <th>Correct</th>
                </tr>
            </thead>

            <tbody>
                {data_type_rows_html}
            </tbody>

        </table>

    </div>

</section>

<!-- DATA_TYPE_CHECK_END -->
"""


# ============================================================
# 5. Insert the table into data_quality.html
# Section 20 is preserved
# ============================================================

data_quality_path = Path("data_quality.html")

if not data_quality_path.exists():
    raise FileNotFoundError(
        "data_quality.html was not found. "
        "Run the previous Data Quality cells first."
    )

current_html = data_quality_path.read_text(
    encoding="utf-8"
)

start_marker = "<!-- DATA_TYPE_CHECK_START -->"
end_marker = "<!-- DATA_TYPE_CHECK_END -->"

# Remove a prior generated copy of this four-column section.
if start_marker in current_html and end_marker in current_html:
    before_section = current_html.split(
        start_marker,
        1,
    )[0]

    after_section = current_html.split(
        end_marker,
        1,
    )[1]

    current_html = before_section + after_section


# Remove the old Data Type Check card that contained
# RAW DTYPE / TYPED DTYPE / CONVERTED, if present.
old_title_candidates = [
    "<h1>DATA TYPE CHECK</h1>",
    "<h2>DATA TYPE CHECK</h2>",
]

for old_title in old_title_candidates:

    if old_title not in current_html:
        continue

    title_position = current_html.index(old_title)

    old_start = current_html.rfind(
        "<section",
        0,
        title_position,
    )

    old_end = current_html.find(
        "</section>",
        title_position,
    )

    if old_start != -1 and old_end != -1:
        current_html = (
            current_html[:old_start]
            + current_html[
                old_end + len("</section>"):
            ]
        )

        break


# Insert this table immediately before Section 20.
ratio_marker = "<!-- FUNNEL_RATIO_VALIDATION_START -->"

if ratio_marker in current_html:

    updated_html = current_html.replace(
        ratio_marker,
        data_type_section_html
        + "\n"
        + ratio_marker,
        1,
    )

elif "</body>" in current_html:

    updated_html = current_html.replace(
        "</body>",
        data_type_section_html
        + "\n</body>",
        1,
    )

else:

    updated_html = (
        current_html
        + data_type_section_html
    )


data_quality_path.write_text(
    updated_html,
    encoding="utf-8",
)


print(
    f"Updated: {data_quality_path.resolve()}"
)

print(
    "Data Type Check restored:"
)

print(
    "METRICS | CURRENT DATA | EXPECTED DATA | CORRECT"
)

print(
    "Section 20 was preserved."
)


data_type_table


Updated: /Users/elvamariaramosgonzalez/Desktop/dashboard-demo/data_quality.html
Data Type Check restored:
METRICS | CURRENT DATA | EXPECTED DATA | CORRECT
Section 20 was preserved.


,METRICS,CURRENT DATA,EXPECTED DATA,CORRECT
0,MONTH,str,date,No
1,CITY,str,str,Yes
2,TRIPS,str,int,No
3,GROSS BOOKINGS (GB),str,int,No
4,SHOPPING SESSIONS,str,int,No
5,RIDERS WITH SESSIONS,str,int,No
6,REQUESTING SESSIONS,str,int,No
7,REQUESTS,str,int,No
8,ACTIVE DRIVERS,str,int,No
9,SUPPLY HOURS,str,int,No


## 20. Funnel Ratio Validation — Row-Level Detail

For every `MONTH + CITY + METRIC`, this section recalculates the reported funnel ratio, calculates the absolute difference, and marks `CORRECT` as `True` when the difference is less than or equal to `0.001`. It then updates `data_quality.html`.


In [116]:
# ============================================================
# FUNNEL RATIO VALIDATION — ROW-LEVEL DETAIL
# ============================================================

from pathlib import Path
import html
import json
import numpy as np
import pandas as pd

ratio_tolerance = 0.001

ratio_definitions = {
    "C/R": {
        "numerator": "TRIPS",
        "denominator": "REQUESTS",
        "formula": "TRIPS / REQUESTS",
    },
    "C/RS": {
        "numerator": "TRIPS",
        "denominator": "REQUESTING SESSIONS",
        "formula": "TRIPS / REQUESTING SESSIONS",
    },
    "RS/SS": {
        "numerator": "REQUESTING SESSIONS",
        "denominator": "SHOPPING SESSIONS",
        "formula": "REQUESTING SESSIONS / SHOPPING SESSIONS",
    },
    "C/SS": {
        "numerator": "TRIPS",
        "denominator": "SHOPPING SESSIONS",
        "formula": "TRIPS / SHOPPING SESSIONS",
    },
}

required_ratio_columns = {
    "MONTH",
    "CITY",
    "TRIPS",
    "REQUESTS",
    "REQUESTING SESSIONS",
    "SHOPPING SESSIONS",
    "C/R",
    "C/RS",
    "RS/SS",
    "C/SS",
}

missing_ratio_columns = sorted(
    required_ratio_columns - set(df_quality.columns)
)

if missing_ratio_columns:
    raise KeyError(
        f"Missing columns required for ratio validation: {missing_ratio_columns}"
    )


def parse_numeric_for_validation(series):
    """Parse numeric values without modifying df_quality."""
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")

    cleaned = (
        series.astype("string")
        .str.strip()
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
    )

    return pd.to_numeric(cleaned, errors="coerce")


def parse_ratio_for_validation(series):
    """Parse ratios stored as decimals or percentages."""
    if pd.api.types.is_numeric_dtype(series):
        numeric = pd.to_numeric(series, errors="coerce")
        return numeric.where(numeric.abs() <= 1, numeric / 100)

    raw = series.astype("string").str.strip()
    contains_percent = raw.str.contains("%", na=False)

    cleaned = (
        raw.str.replace("%", "", regex=False)
        .str.replace(",", "", regex=False)
    )

    numeric = pd.to_numeric(cleaned, errors="coerce")
    numeric = numeric.where(~contains_percent, numeric / 100)
    numeric = numeric.where(numeric.abs() <= 1, numeric / 100)

    return numeric


validation_frames = []

for metric, definition in ratio_definitions.items():
    numerator = parse_numeric_for_validation(
        df_quality[definition["numerator"]]
    )

    denominator = parse_numeric_for_validation(
        df_quality[definition["denominator"]]
    )

    reported = parse_ratio_for_validation(
        df_quality[metric]
    )

    recalculated = numerator.div(
        denominator.where(denominator.ne(0))
    )

    difference = (reported - recalculated).abs()

    correct = difference.le(ratio_tolerance).astype("boolean")
    correct = correct.where(
        reported.notna() & recalculated.notna(),
        pd.NA,
    )

    validation_frames.append(
        pd.DataFrame({
            "MONTH": df_quality["MONTH"],
            "CITY": df_quality["CITY"],
            "METRIC": metric,
            "FORMULA": definition["formula"],
            "REPORTED": reported,
            "RECALCULATED": recalculated,
            "DIFFERENCE": difference,
            "CORRECT": correct,
        })
    )

ratio_validation_detail = pd.concat(
    validation_frames,
    ignore_index=True,
)

# Preserve a stable initial order.
ratio_validation_detail = ratio_validation_detail.sort_values(
    ["MONTH", "CITY", "METRIC"],
    kind="stable",
).reset_index(drop=True)

# Summary metrics.
testable_mask = ratio_validation_detail["CORRECT"].notna()
total_validations = int(testable_mask.sum())
correct_validations = int(
    ratio_validation_detail.loc[testable_mask, "CORRECT"].eq(True).sum()
)
incorrect_validations = int(
    ratio_validation_detail.loc[testable_mask, "CORRECT"].eq(False).sum()
)
validation_accuracy = (
    correct_validations / total_validations
    if total_validations > 0
    else np.nan
)

# Filter options.
month_options = sorted(
    ratio_validation_detail["MONTH"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

city_options = sorted(
    ratio_validation_detail["CITY"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

metric_options = list(ratio_definitions.keys())


def build_select_options(values):
    return "".join(
        f'<option value="{html.escape(value)}">{html.escape(value)}</option>'
        for value in values
    )


month_options_html = build_select_options(month_options)
city_options_html = build_select_options(city_options)
metric_options_html = build_select_options(metric_options)

# Build detailed HTML rows.
ratio_rows_html = ""

for _, row in ratio_validation_detail.iterrows():
    if pd.isna(row["CORRECT"]):
        correct_label = "N/A"
        correct_class = "ratio-na"
    elif bool(row["CORRECT"]):
        correct_label = "True"
        correct_class = "ratio-true"
    else:
        correct_label = "False"
        correct_class = "ratio-false"

    reported_label = (
        f'{row["REPORTED"]:.4f}'
        if pd.notna(row["REPORTED"])
        else "N/A"
    )

    recalculated_label = (
        f'{row["RECALCULATED"]:.4f}'
        if pd.notna(row["RECALCULATED"])
        else "N/A"
    )

    difference_label = (
        f'{row["DIFFERENCE"]:.4f}'
        if pd.notna(row["DIFFERENCE"])
        else "N/A"
    )

    month_value = html.escape(str(row["MONTH"]))
    city_value = html.escape(str(row["CITY"]))
    metric_value = html.escape(str(row["METRIC"]))

    ratio_rows_html += f"""
    <tr
        data-month="{month_value}"
        data-city="{city_value}"
        data-metric="{metric_value}"
        data-correct="{correct_label}"
    >
        <td data-sort-value="{month_value}">{month_value}</td>
        <td data-sort-value="{city_value}">{city_value}</td>
        <td class="ratio-metric" data-sort-value="{metric_value}">
            {metric_value}
        </td>
        <td>{html.escape(str(row['FORMULA']))}</td>
        <td data-sort-value="{reported_label}">{reported_label}</td>
        <td data-sort-value="{recalculated_label}">{recalculated_label}</td>
        <td data-sort-value="{difference_label}">{difference_label}</td>
        <td data-sort-value="{correct_label}">
            <span class="ratio-badge {correct_class}">{correct_label}</span>
        </td>
    </tr>
    """

accuracy_label = (
    f"{validation_accuracy:.1%}"
    if pd.notna(validation_accuracy)
    else "N/A"
)

ratio_section_html = f"""
<!-- FUNNEL_RATIO_VALIDATION_START -->
<style>
    .ratio-section {{
        max-width: 1400px;
        margin: 24px auto 0;
        background: var(--surface);
        border: 1px solid var(--border);
        border-radius: var(--radius);
        box-shadow: 0 1px 2px rgba(18, 18, 18, 0.04);
        padding: 28px 32px;
        color: var(--ink);
        font-family: var(--sans);
    }}

    .ratio-section h2 {{
        text-align: center;
        margin: -28px -32px 22px;
        font-size: 21px;
    }}

    .ratio-description {{
        margin: 0 0 20px 14px;
        text-align: left;
        color: var(--muted);
        font-size: 13px;
        line-height: 1.5;
    }}

    .ratio-disclaimer {{
        max-width: 980px;
        margin: 0 0 24px;
        padding: 14px 16px;
        border: 1px solid var(--border);
        border-left: 4px solid var(--blue);
        border-radius: var(--radius-sm);
        background: var(--canvas);
        color: var(--muted);
        font-size: 13px;
        line-height: 1.55;
        text-align: left;
    }}

    .ratio-disclaimer strong {{
        color: var(--ink);
    }}

    .ratio-summary-grid {{
        display: grid;
        grid-template-columns: repeat(4, minmax(0, 1fr));
        gap: 16px;
        margin-bottom: 24px;
    }}

    .ratio-summary-card {{
        border: 1px solid var(--ink);
        border-left: 4px solid var(--blue);
        border-radius: var(--radius-sm);
        padding: 16px 18px;
        text-align: left;
        background: var(--surface);
    }}

    .ratio-summary-label {{
        color: var(--muted);
        font-size: 11px;
        font-weight: 700;
        text-transform: uppercase;
        letter-spacing: 0.45px;
        margin-bottom: 8px;
    }}

    .ratio-summary-value {{
        font-family: var(--mono);
        font-size: 22px;
        font-weight: 700;
        color: var(--ink);
    }}

    .ratio-filter-grid {{
        display: grid;
        grid-template-columns: repeat(5, minmax(0, 1fr));
        gap: 14px;
        margin-bottom: 16px;
        align-items: stretch;
    }}

    .ratio-filter-group {{
        display: flex;
        flex-direction: column;
        justify-content: flex-end;
    }}

    .ratio-filter-group label {{
        display: block;
        margin-bottom: 6px;
        color: var(--muted);
        font-size: 11px;
        font-weight: 700;
        text-transform: uppercase;
        letter-spacing: 0.35px;
    }}

    .ratio-filter-group select,
    .ratio-filter-group input {{
        width: 100%;
        height: 38px;
        border: 1px solid var(--border);
        border-radius: var(--radius-sm);
        padding: 0 11px;
        background: var(--surface);
        color: var(--ink);
        font-size: 13px;
        font-family: var(--sans);
    }}

    .ratio-reset-button {{
        width: 100%;
        height: 38px;
        border: 1px solid var(--ink);
        border-radius: var(--radius-sm);
        background: var(--ink);
        color: #FFFFFF;
        font-size: 13px;
        font-weight: 700;
        text-transform: uppercase;
        letter-spacing: 0.03em;
        cursor: pointer;
    }}

    .ratio-reset-button:hover {{
        background: var(--blue);
        border-color: var(--blue);
        color: #FFFFFF;
    }}

    .ratio-visible-count {{
        margin: 0 0 10px;
        color: var(--muted);
        font-size: 12px;
        text-align: right;
        font-family: var(--mono);
    }}

    .ratio-table-wrapper {{
        overflow: auto;
        max-height: 720px;
        border: 1px solid var(--border);
        border-radius: var(--radius-sm);
    }}

    .ratio-table {{
        width: 100%;
        border-collapse: collapse;
        min-width: 950px;
    }}

    .ratio-table th {{
        position: sticky;
        top: 0;
        background: var(--ink);
        color: #FFFFFF;
        text-transform: uppercase;
        font-size: 11px;
        letter-spacing: 0.45px;
        padding: 14px 16px;
        text-align: left;
        border-bottom: 1px solid var(--ink);
        z-index: 1;
        user-select: none;
    }}

    .ratio-table th[data-sort-index] {{
        cursor: pointer;
    }}

    .ratio-table th[data-sort-index]:hover {{
        background: var(--blue);
        color: #FFFFFF;
    }}
    .ratio-table th[data-sort-index]:hover .ratio-sort-indicator {{
        color: #FFFFFF;
    }}

    .ratio-sort-indicator {{
        margin-left: 5px;
        font-size: 10px;
        color: var(--blue);
    }}

    .ratio-table td {{
        padding: 13px 16px;
        border-bottom: 1px solid var(--border);
        font-size: 13px;
        white-space: nowrap;
    }}

    .ratio-table tbody tr:hover {{
        background: var(--blue-tint);
    }}

    .ratio-metric {{
        font-weight: 700;
    }}

    .ratio-table td:nth-child(5),
    .ratio-table td:nth-child(6),
    .ratio-table td:nth-child(7) {{
        font-family: var(--mono);
        color: var(--ink);
    }}

    .ratio-badge {{
        display: inline-block;
        padding: 4px 10px 4px 8px;
        border-radius: 4px;
        font-size: 11px;
        font-weight: 700;
        text-transform: uppercase;
        letter-spacing: 0.03em;
    }}

    .ratio-true {{
        background: var(--ok-bg);
        color: var(--ok);
        border-left: 3px solid var(--ok);
    }}

    .ratio-false {{
        background: var(--bad-bg);
        color: var(--bad);
        border-left: 3px solid var(--bad);
    }}

    .ratio-na {{
        background: #ECEDEF;
        color: var(--muted);
        border-left: 3px solid var(--faint);
    }}

    @media (max-width: 1050px) {{
        .ratio-filter-grid {{
            grid-template-columns: repeat(2, minmax(0, 1fr));
        }}
    }}

    @media (max-width: 900px) {{
        .ratio-summary-grid {{
            grid-template-columns: repeat(2, minmax(0, 1fr));
        }}
    }}

    @media (max-width: 620px) {{
        .ratio-filter-grid {{
            grid-template-columns: 1fr;
        }}
    }}
</style>

<section class="ratio-section" id="funnel-ratio-validation">
    <h2>FUNNEL RATIO VALIDATION</h2>

    <p class="ratio-description">
        This validation recalculates C/R, C/RS, RS/SS, and C/SS for every
        row, then compares each result with the reported value. It shows
        the absolute difference and marks the result as correct when the
        difference is {ratio_tolerance:.3f} or less. Use the filters below
        or select a column header to sort the table.
    </p>

    <div class="ratio-disclaimer">
        <strong>Validation tolerance: 0.001.</strong>
        This threshold is applied because the reported compound metrics
        may include rounding during calculation or storage. Minor
        differences between reported and recalculated values are therefore
        treated as acceptable rather than as data-quality failures.
    </div>

    <div class="ratio-summary-grid">
        <div class="ratio-summary-card">
            <div class="ratio-summary-label">Total Validations</div>
            <div class="ratio-summary-value">{total_validations:,}</div>
        </div>

        <div class="ratio-summary-card">
            <div class="ratio-summary-label">Correct</div>
            <div class="ratio-summary-value">{correct_validations:,}</div>
        </div>

        <div class="ratio-summary-card">
            <div class="ratio-summary-label">Incorrect</div>
            <div class="ratio-summary-value">{incorrect_validations:,}</div>
        </div>

        <div class="ratio-summary-card">
            <div class="ratio-summary-label">Accuracy</div>
            <div class="ratio-summary-value">{accuracy_label}</div>
        </div>
    </div>

    <div class="ratio-filter-grid">
        <div class="ratio-filter-group">
            <label for="ratio-month-filter">Month</label>
            <select id="ratio-month-filter">
                <option value="">All months</option>
                {month_options_html}
            </select>
        </div>

        <div class="ratio-filter-group">
            <label for="ratio-city-filter">City</label>
            <select id="ratio-city-filter">
                <option value="">All cities</option>
                {city_options_html}
            </select>
        </div>

        <div class="ratio-filter-group">
            <label for="ratio-metric-filter">Metric</label>
            <select id="ratio-metric-filter">
                <option value="">All metrics</option>
                {metric_options_html}
            </select>
        </div>

        <div class="ratio-filter-group">
            <label for="ratio-correct-filter">Correct</label>
            <select id="ratio-correct-filter">
                <option value="">All results</option>
                <option value="True">True</option>
                <option value="False">False</option>
                <option value="N/A">N/A</option>
            </select>
        </div>

        <div class="ratio-filter-group">
            <button
                type="button"
                id="ratio-reset-filters"
                class="ratio-reset-button"
            >
                Reset filters
            </button>
        </div>
    </div>

    <div id="ratio-visible-count" class="ratio-visible-count"></div>

    <div class="ratio-table-wrapper">
        <table id="ratio-validation-table" class="ratio-table">
            <thead>
                <tr>
                    <th data-sort-index="0" data-sort-type="text">
                        Month<span class="ratio-sort-indicator"></span>
                    </th>
                    <th data-sort-index="1" data-sort-type="text">
                        City<span class="ratio-sort-indicator"></span>
                    </th>
                    <th data-sort-index="2" data-sort-type="text">
                        Metric<span class="ratio-sort-indicator"></span>
                    </th>
                    <th>Formula</th>
                    <th data-sort-index="4" data-sort-type="number">
                        Reported<span class="ratio-sort-indicator"></span>
                    </th>
                    <th data-sort-index="5" data-sort-type="number">
                        Recalculated<span class="ratio-sort-indicator"></span>
                    </th>
                    <th data-sort-index="6" data-sort-type="number">
                        Difference<span class="ratio-sort-indicator"></span>
                    </th>
                    <th data-sort-index="7" data-sort-type="text">
                        Correct<span class="ratio-sort-indicator"></span>
                    </th>
                </tr>
            </thead>
            <tbody>
                {ratio_rows_html}
            </tbody>
        </table>
    </div>
</section>

<script>
(function () {{
    const table = document.getElementById("ratio-validation-table");

    if (!table) {{
        return;
    }}

    const tbody = table.querySelector("tbody");
    const monthFilter = document.getElementById("ratio-month-filter");
    const cityFilter = document.getElementById("ratio-city-filter");
    const metricFilter = document.getElementById("ratio-metric-filter");
    const correctFilter = document.getElementById("ratio-correct-filter");
    const resetButton = document.getElementById("ratio-reset-filters");
    const visibleCount = document.getElementById("ratio-visible-count");

    let sortIndex = null;
    let sortDirection = "asc";

    function applyFilters() {{
        const monthValue = monthFilter.value;
        const cityValue = cityFilter.value;
        const metricValue = metricFilter.value;
        const correctValue = correctFilter.value;

        let visibleRows = 0;

        Array.from(tbody.rows).forEach((row) => {{
            const matchesMonth =
                !monthValue || row.dataset.month === monthValue;

            const matchesCity =
                !cityValue || row.dataset.city === cityValue;

            const matchesMetric =
                !metricValue || row.dataset.metric === metricValue;

            const matchesCorrect =
                !correctValue || row.dataset.correct === correctValue;

            const visible =
                matchesMonth
                && matchesCity
                && matchesMetric
                && matchesCorrect;

            row.style.display = visible ? "" : "none";

            if (visible) {{
                visibleRows += 1;
            }}
        }});

        visibleCount.textContent =
            `Showing ${{visibleRows.toLocaleString()}} of `
            + `${{tbody.rows.length.toLocaleString()}} validations`;
    }}

    function getCellValue(row, index, type) {{
        const cell = row.cells[index];
        const rawValue =
            cell.dataset.sortValue || cell.textContent.trim();

        if (type === "number") {{
            const numericValue = Number(rawValue);
            return Number.isFinite(numericValue)
                ? numericValue
                : Number.POSITIVE_INFINITY;
        }}

        return rawValue.toLowerCase();
    }}

    function sortTable(header) {{
        const nextSortIndex = Number(header.dataset.sortIndex);
        const sortType = header.dataset.sortType || "text";

        if (sortIndex === nextSortIndex) {{
            sortDirection =
                sortDirection === "asc" ? "desc" : "asc";
        }} else {{
            sortIndex = nextSortIndex;
            sortDirection = "asc";
        }}

        const rows = Array.from(tbody.rows);

        rows.sort((rowA, rowB) => {{
            const valueA = getCellValue(
                rowA,
                sortIndex,
                sortType,
            );

            const valueB = getCellValue(
                rowB,
                sortIndex,
                sortType,
            );

            if (valueA < valueB) {{
                return sortDirection === "asc" ? -1 : 1;
            }}

            if (valueA > valueB) {{
                return sortDirection === "asc" ? 1 : -1;
            }}

            return 0;
        }});

        rows.forEach((row) => tbody.appendChild(row));

        table.querySelectorAll(".ratio-sort-indicator")
            .forEach((indicator) => {{
                indicator.textContent = "";
            }});

        const indicator = header.querySelector(
            ".ratio-sort-indicator"
        );

        if (indicator) {{
            indicator.textContent =
                sortDirection === "asc" ? "▲" : "▼";
        }}
    }}

    [
        monthFilter,
        cityFilter,
        metricFilter,
        correctFilter,
    ].forEach((filter) => {{
        filter.addEventListener("change", applyFilters);
    }});

    resetButton.addEventListener("click", () => {{
        monthFilter.value = "";
        cityFilter.value = "";
        metricFilter.value = "";
        correctFilter.value = "";
        applyFilters();
    }});

    table.querySelectorAll("th[data-sort-index]")
        .forEach((header) => {{
            header.addEventListener("click", () => {{
                sortTable(header);
            }});
        }});

    applyFilters();
}})();
</script>
<!-- FUNNEL_RATIO_VALIDATION_END -->
"""

# Update the HTML generated by the prior notebook sections.
data_quality_path = Path("data_quality.html")

if not data_quality_path.exists():
    raise FileNotFoundError(
        "data_quality.html was not found. Run the prior Data Quality cells first."
    )

current_html = data_quality_path.read_text(encoding="utf-8")

start_marker = "<!-- FUNNEL_RATIO_VALIDATION_START -->"
end_marker = "<!-- FUNNEL_RATIO_VALIDATION_END -->"

if start_marker in current_html and end_marker in current_html:
    before = current_html.split(start_marker, 1)[0]
    after = current_html.split(end_marker, 1)[1]
    current_html = before + after

# The .wrap div opened in the Data Nulls cell is only closed here, after this
# last section. This sentinel makes the wrap-closing div itself removable on
# reruns, so re-executing this cell never leaves a duplicate closing </div>.
wrap_close_start = "<!-- WRAP_CLOSE_START -->"
wrap_close_end = "<!-- WRAP_CLOSE_END -->"

if wrap_close_start in current_html and wrap_close_end in current_html:
    before_wc = current_html.split(wrap_close_start, 1)[0]
    after_wc = current_html.split(wrap_close_end, 1)[1]
    current_html = before_wc + after_wc

wrap_close_html = wrap_close_start + "\n  </div>\n" + wrap_close_end

if "</body>" in current_html:
    updated_html = current_html.replace(
        "</body>",
        ratio_section_html + "\n" + wrap_close_html + "\n</body>",
        1,
    )
else:
    updated_html = current_html + ratio_section_html + "\n" + wrap_close_html

data_quality_path.write_text(
    updated_html,
    encoding="utf-8",
)

print(f"Updated: {data_quality_path.resolve()}")
print(f"Tolerance: {ratio_tolerance}")
print(f"Total validations: {total_validations:,}")
print(f"Correct: {correct_validations:,}")
print(f"Incorrect: {incorrect_validations:,}")
print(f"Accuracy: {accuracy_label}")
print("Filters: MONTH, CITY, METRIC, CORRECT")
print("Sorting: click the sortable table headers")

ratio_validation_detail.head(12)


Updated: /Users/elvamariaramosgonzalez/Desktop/dashboard-demo/data_quality.html
Tolerance: 0.001
Total validations: 672
Correct: 672
Incorrect: 0
Accuracy: 100.0%
Filters: MONTH, CITY, METRIC, CORRECT
Sorting: click the sortable table headers


,MONTH,CITY,METRIC,FORMULA,REPORTED,RECALCULATED,DIFFERENCE,CORRECT
0,2022-01,Azteca,C/R,TRIPS / REQUESTS,0.696,0.695994,0.000006,True
1,2022-01,Azteca,C/RS,TRIPS / REQUESTING SESSIONS,0.87,0.86983,0.00017,True
2,2022-01,Azteca,C/SS,TRIPS / SHOPPING SESSIONS,0.493,0.492691,0.000309,True
3,2022-01,Azteca,RS/SS,REQUESTING SESSIONS / SHOPPING SESSIONS,0.566,0.566423,0.000423,True
4,2022-01,Lusail,C/R,TRIPS / REQUESTS,0.757,0.756607,0.000393,True
5,2022-01,Lusail,C/RS,TRIPS / REQUESTING SESSIONS,0.921,0.921375,0.000375,True
6,2022-01,Lusail,C/SS,TRIPS / SHOPPING SESSIONS,0.549,0.549163,0.000163,True
7,2022-01,Lusail,RS/SS,REQUESTING SESSIONS / SHOPPING SESSIONS,0.596,0.596025,0.000025,True
8,2022-01,Maracanã,C/R,TRIPS / REQUESTS,0.761,0.761401,0.000401,True
9,2022-01,Maracanã,C/RS,TRIPS / REQUESTING SESSIONS,0.909,0.909126,0.000126,True
